# Train `partwise` — Sheet `v4 22/11/26` (Colab — Google Drive Mount)
Mở notebook trong Google Colab và bấm **Run all (Chạy tất cả)** hoặc chạy lần lượt từ trên xuống.
- **Ưu tiên đọc từ Drive đã mount:** ánh xạ File ID trên sheet sang `/content/drive`; file không tìm thấy trên mount có thể được tải qua API/gdown. Với dữ liệu trong **Được chia sẻ với tôi**, thêm lối tắt thư mục vào `MyDrive` trước khi chạy.
- Loss có phạt theo frame khi PA hoặc MPJPE thua naive. **Best Epoch** ưu tiên checkpoint thắng naive ở cả PA và MPJPE validation (12 khớp thân), rồi mới xét PA + λ·MPJPE. Nếu chưa có checkpoint đạt, vẫn lưu ứng viên tốt nhất và ghi `val_beats_naive: false`.
- Lưu **mọi epoch** trên **Google Drive**, có giới hạn epoch và dừng sớm khi không cải thiện; xuất `best_weights.json`, `best_partwise.yml`, `pipeline_best.yml`. Thắng validation chưa thay thế benchmark độc lập; không dùng test để ép train đến khi thắng.

**Lưu ý phạm vi:** GT chỉ dùng khi chuẩn bị dữ liệu giám sát (ghép track/thời gian và tính hàm mục tiêu), **0% GT lúc suy luận**. Kiểm tra bảng `alignment.json` ở Bước 4a trước khi dùng bộ tham số.


In [ ]:
#@title 1. Cài đặt môi trường & clone repo
import json, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
BRANCH = "experiment/observability-aware-trust-region"  #@param {type:"string"}
REPO_DIR = Path("/content/optimization_model_monocular_3")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-f", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "chumpy"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2.3", "scipy", "torch", "joblib", "pyyaml", "trimesh",
                "gspread", "google-api-python-client", "gdown", "matplotlib", "pandas"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def run_live(cmd):
    # ponytail: stream child stdout live into Colab cell so progress & tracebacks are never hidden
    lines = []
    with subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
        rc = proc.wait()
    if rc:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}\n" + "".join(lines[-40:]))
    return lines

print("Ready | repo:", REPO_DIR, "| branch:", BRANCH)


In [ ]:
#@title 2. Đồng bộ snapshot trainer & chạy self-test
SNAPSHOTS = {'compat.py': 'import sys\nimport copy\nimport numpy as np\nimport joblib\nimport inspect\n\nimport warnings\n\ndef patch_numpy_and_inspect():\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore", FutureWarning)\n        if not hasattr(np, \'float\'): np.float = float\n        if not hasattr(np, \'int\'): np.int = int\n        if not hasattr(np, \'bool\'): np.bool = bool\n        if not hasattr(np, \'object\'): np.object = object\n        if not hasattr(np, \'typeDict\'): np.typeDict = np.sctypeDict\n        if not hasattr(np, \'complex\'): np.complex = complex\n        if not hasattr(np, \'unicode\'): np.unicode = str\n        if not hasattr(np, \'str\'): np.str = str\n    if not hasattr(inspect, \'getargspec\'): inspect.getargspec = inspect.getfullargspec\n\ndef configure_stdout_encoding():\n    if hasattr(sys.stdout, \'reconfigure\'):\n        sys.stdout.reconfigure(encoding=\'utf-8\')\n\nfrom functools import lru_cache\nimport os\nfrom pathlib import Path\n\n\n@lru_cache(maxsize=4)\ndef _load_joblib_cached(abs_path, mtime_ns, size):\n    # ponytail: cache 4 hot pkl/pth files per process (each pair loads same pkls ~5x); keyed by mtime+size\n    with open(abs_path, \'rb\') as f:\n        return joblib.load(f)\n\n\ndef load_joblib_compat(filepath):\n    abs_path = os.path.abspath(str(filepath))\n    st = os.stat(abs_path)\n    return _load_joblib_cached(abs_path, st.st_mtime_ns, st.st_size)\n\n\ndef normalize_person_name(target_person):\n    if target_person is None:\n        return None\n    s = str(target_person).strip().lower()\n    if s in ("aria1", "aria_1", "aria-1"):\n        return "aria01"\n    if s in ("aria2", "aria_2", "aria-2"):\n        return "aria02"\n    return s\n\n\ndef parse_person_mapping(pkl_path):\n    if pkl_path is None:\n        return {}\n    #track id -> person for one WHAM pkl, from its own folder only.\n\n    """`<pkl stem>_person_mapping.txt` wins; otherwise the folder\'s single *person_mapping.txt.\n    Several mappings and none named after this pkl -> error. Parent folders are never searched:\n    that used to hand one camera\'s labels to every pkl of a shared folder (wrong identity)."""\n    pkl_path = Path(pkl_path)\n    own = pkl_path.parent / f"{pkl_path.stem}_person_mapping.txt"\n    if own.is_file():\n        candidates = [own]\n    else:\n        candidates = sorted(pkl_path.parent.glob("*person_mapping.txt"))\n        if len(candidates) > 1:\n            raise ValueError(\n                f"{len(candidates)} person mappings next to {pkl_path.name} and none is "\n                f"{own.name}: {[c.name for c in candidates]}. Keep one WHAM pkl per folder "\n                "(exo/<cam>/<seq>_<cam>_wham/) or name the mapping after the pkl.")\n    mapping = {}\n    if candidates:\n        for line in candidates[0].read_text(encoding="utf-8").splitlines():\n            if "=" in line:\n                k, v = line.split("=", 1)\n                mapping[k.strip()] = v.strip().lower()\n    return mapping\n\n\ndef resolve_target_track_ids(wham_data, pkl_path=None, target_person=None):\n    """Return the set of track keys in wham_data that belong to target_person."""\n    if not isinstance(wham_data, dict) and "defaultdict" not in str(type(wham_data)):\n        return set()\n    if target_person is not None:\n        target_str = normalize_person_name(target_person)\n        if pkl_path is not None:\n            mapping = parse_person_mapping(pkl_path)\n            found = set()\n            for track_str, person_name in mapping.items():\n                if person_name == target_str:\n                    track_id = int(track_str) if track_str.isdigit() else track_str\n                    if track_id in wham_data:\n                        found.add(track_id)\n                    elif str(track_id) in wham_data:\n                        found.add(str(track_id))\n            if found or mapping:\n                return found\n        for cand in (target_person, target_str, int(target_str) if target_str.isdigit() else None):\n            if cand is not None and cand in wham_data:\n                return {cand}\n        if target_str.startswith("s") and target_str[1:].isdigit() and 0 in wham_data:\n            return {0}\n        return set()\n    if 0 in wham_data:\n        return {0}\n    if "0" in wham_data:\n        return {"0"}\n    best_key, max_frames = None, -1\n    for k, v in wham_data.items():\n        if isinstance(v, dict) and "pose" in v and len(v["pose"]) > max_frames:\n            best_key, max_frames = k, len(v["pose"])\n    return {best_key} if best_key is not None else set()\n\n\ndef _payload_frame_ids(payload):\n    frame_ids = payload.get("frame_ids")\n    per_frame = payload.get("pose", [])\n    if frame_ids is None and isinstance(payload.get("tracking_results_for_reproj"), dict):\n        frame_ids = payload["tracking_results_for_reproj"].get("frame_id")\n    if frame_ids is None and payload.get("frame_id") is not None:\n        # A track of WHAM\'s tracking_results.pth: frame_id + keypoints (2D with confidence).\n        frame_ids, per_frame = payload["frame_id"], payload.get("keypoints", [])\n    if frame_ids is None:\n        return None\n    if hasattr(frame_ids, "detach"):  # torch tensor\n        frame_ids = frame_ids.detach().cpu().numpy()\n    frame_ids = np.asarray(frame_ids).astype(int).reshape(-1)\n    return frame_ids if per_frame is not None and len(frame_ids) == len(per_frame) else None\n\n\ndef _frame_count(payload):\n    ids = _payload_frame_ids(payload)\n    return len(ids) if ids is not None else len(payload.get("pose", []))\n\n\ndef _merge_frame_value(values, lengths, rows):\n    if any(value is None for value in values):\n        return None\n    if all(isinstance(value, dict) for value in values):\n        merged = {}\n        for key in values[0]:\n            if all(key in value for value in values):\n                item = _merge_frame_value([value[key] for value in values], lengths, rows)\n                if item is not None:\n                    merged[key] = item\n        return merged or None\n    if not all(hasattr(value, "__len__") and len(value) == length for value, length in zip(values, lengths)):\n        return None\n    selected = [values[candidate][row] for candidate, row in rows]\n    if all(isinstance(value, np.ndarray) for value in values):\n        return np.asarray(selected)\n    if all(isinstance(value, list) for value in values):\n        return selected\n    try:\n        import torch\n        if all(torch.is_tensor(value) for value in values):\n            return torch.stack(selected)\n    except ImportError:\n        pass\n    return None\n\n\ndef _merge_person_tracks(candidates):\n    """Join re-ID fragments by absolute frame id; the longest track wins overlaps.\n    Works for WHAM pkl tracks (frame_ids / pose) and tracking_results.pth tracks\n    (frame_id / keypoints), so every fragment of a person keeps its 2D keypoints."""\n    candidates = sorted(candidates, key=_frame_count, reverse=True)\n    frame_ids = [_payload_frame_ids(value) for value in candidates]\n    if len(candidates) == 1 or any(ids is None for ids in frame_ids):\n        return candidates[0]\n\n    chosen = {}\n    for candidate, ids in enumerate(frame_ids):\n        for row, frame_id in enumerate(ids):\n            chosen.setdefault(int(frame_id), (candidate, row))\n    ordered_ids = sorted(chosen)\n    rows = [chosen[frame_id] for frame_id in ordered_ids]\n    lengths = [len(ids) for ids in frame_ids]\n    merged = copy.deepcopy(candidates[0])\n    for key in set.intersection(*(set(value) for value in candidates)):\n        value = _merge_frame_value([candidate[key] for candidate in candidates], lengths, rows)\n        if value is not None:\n            merged[key] = value\n    merged["frame_ids"] = np.asarray(ordered_ids)\n    return merged\n\n\ndef resolve_person_payload(wham_data, pkl_path=None, target_person=None):\n    if not isinstance(wham_data, dict) and "defaultdict" not in str(type(wham_data)):\n        if isinstance(wham_data, list):\n            for v in wham_data:\n                if isinstance(v, dict):\n                    return v\n        return None\n\n    if target_person is not None:\n        target_str = str(target_person).strip().lower()\n        if target_str in ("aria1", "aria_1", "aria-1"):\n            target_str = "aria01"\n        elif target_str in ("aria2", "aria_2", "aria-2"):\n            target_str = "aria02"\n\n        if pkl_path is not None:\n            mapping = parse_person_mapping(pkl_path)\n            candidate_tracks = []\n            for track_str, person_name in mapping.items():\n                if person_name == target_str:\n                    track_id = int(track_str) if track_str.isdigit() else track_str\n                    if track_id in wham_data:\n                        candidate_tracks.append(wham_data[track_id])\n                    elif str(track_id) in wham_data:\n                        candidate_tracks.append(wham_data[str(track_id)])\n            if candidate_tracks:\n                return _merge_person_tracks(candidate_tracks)\n            if mapping:\n                # ponytail: strict identity - mapping exists but target is missing, never substitute another person\n                return None\n\n        if target_person in wham_data:\n            return wham_data[target_person]\n        if target_str in wham_data:\n            return wham_data[target_str]\n        if target_str.isdigit() and int(target_str) in wham_data:\n            return wham_data[int(target_str)]\n        if target_str.startswith("s") and target_str[1:].isdigit() and 0 in wham_data:\n            return wham_data[0]\n\n        # ponytail: strict identity - do NOT guess or substitute another person when target_person is specified\n        return None\n\n    if 0 in wham_data:\n        return wham_data[0]\n    if "0" in wham_data:\n        return wham_data["0"]\n    best_person = None\n    max_frames = -1\n    for k, v in wham_data.items():\n        if isinstance(v, dict) and "pose" in v:\n            flen = len(v["pose"])\n            if flen > max_frames:\n                max_frames = flen\n                best_person = v\n    if best_person is not None:\n        return best_person\n    for v in wham_data.values():\n        if isinstance(v, dict):\n            return v\n    return None\n\n', 'train_v4.py': '"""Supervised calibration of partwise weights from the v4 Sheets export.\n\nGT is used for track/time association and training targets, never for inference.\nAlignment reports must be reviewed before interpreting validation scores.\n"""\nimport argparse\nimport copy\nimport hashlib\nimport itertools\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport torch\nimport yaml\n\nfrom compat import patch_numpy_and_inspect\nfrom fusion_pipeline.aotr import _ordered_bones\nfrom fusion_pipeline.naive import fuse_naive\nfrom fusion_pipeline.partwise import (\n    FEATURE_NAMES, blend_frame, eval_weight_model, extract_pair_features,\n    fuse_partwise, slave_directions,\n)\nfrom keypoints_map import load_keypoints3d_map\n\npatch_numpy_and_inspect()\n\nROOT = Path(__file__).resolve().parent\nBODY = ("left_shoulder", "right_shoulder", "left_elbow", "right_elbow",\n        "left_wrist", "right_wrist", "left_hip", "right_hip",\n        "left_knee", "right_knee", "left_ankle", "right_ankle")\n\n\ndef validate_rows(rows):\n    groups, seen, gt_splits = {}, set(), {}\n    for row in rows:\n        action, cam, split = (str(row.get(k, "")).strip() for k in ("action", "cam", "split"))\n        if not action or not cam.startswith("cam") or not cam[3:].isdigit():\n            raise ValueError(f"Invalid action/camera: {row}")\n        if action.endswith("_" + cam) or split not in {"train", "val"}:\n            raise ValueError(f"Normalize action and split first: {action}/{cam}/{split}")\n        if (action, cam) in seen:\n            raise ValueError(f"Duplicate camera: {action}/{cam}")\n        seen.add((action, cam))\n        if not row.get("file PKL") or not row.get("GT"):\n            raise ValueError(f"Missing PKL/GT: {action}/{cam}")\n        for key, mapping in ((action, groups), (str(row["GT"]), gt_splits)):\n            previous = mapping.setdefault(key, split)\n            if previous != split:\n                raise ValueError(f"Train/val leakage: {key}")\n    if set(groups.values()) != {"train", "val"}:\n        raise ValueError("Both train and val sequences are required")\n    for action in groups:\n        if sum(r["action"] == action for r in rows) < 2:\n            raise ValueError(f"Need two cameras: {action}")\n    return groups\n\n\ndef file_id(value):\n    import re\n    value = str(value).strip()\n    if "/" in value or "?" in value:\n        match = re.search(r"/d/([\\w-]+)|[?&]id=([\\w-]+)", value)\n        if not match:\n            raise ValueError(f"Not a Drive file ID/URL: {value}")\n        return next(x for x in match.groups() if x)\n    if not re.fullmatch(r"[\\w-]{10,}", value):\n        raise ValueError(f"Invalid Drive file ID: {value}")\n    return value\n\n\nSKIP_SCAN_DIRS = frozenset({\n    "images", "images (1)", "color", "depth", "colmap", "sparse", "dense",\n    ".git", "__pycache__", ".ipynb_checkpoints", "node_modules", "Trash", ".Trash",\n})\n\n\ndef _xattr_drive_id(path):\n    if not hasattr(os, "getxattr"):\n        return None\n    try:\n        return os.getxattr(path, "user.drive.id").decode("utf-8").strip()\n    except OSError:\n        return None\n\n\ndef index_mounted_files(wanted_ids, data_dir, exclude_dir=None):\n    """Scan an explicit dataset folder via user.drive.id xattr (never crawls all of MyDrive)."""\n    root = Path(data_dir)\n    if not data_dir or not root.is_dir():\n        return {}\n    wanted = set(wanted_ids)\n    found = {}\n    excl = os.path.realpath(exclude_dir) if exclude_dir and Path(exclude_dir).exists() else None\n    visited = set()\n    for dirpath, dirs, filenames in os.walk(root, followlinks=True):\n        real = os.path.realpath(dirpath)\n        if real in visited or (excl and (real == excl or real.startswith(excl + os.sep))):\n            dirs[:] = []\n            continue\n        visited.add(real)\n        dirs[:] = [d for d in dirs if d not in SKIP_SCAN_DIRS and not d.startswith(".")]\n        for fname in filenames:\n            if not fname.endswith((".pkl", ".npz")):\n                continue\n            full = os.path.join(dirpath, fname)\n            did = _xattr_drive_id(full)\n            if did in wanted:\n                found[did] = Path(full)\n                if len(found) == len(wanted):\n                    return found\n    return found\n\n\ndef resolve_mounted(drive, value, mount, state=None):\n    """Resolve a Drive file ID/URL to a local path on mounted Google Drive (0 media downloads)."""\n    raw = str(value).strip()\n    direct = Path(raw)\n    if direct.is_file():\n        return direct\n    mount = Path(mount)\n    if not mount.exists():\n        raise FileNotFoundError(f"Google Drive chưa được mount tại {mount}; hãy chạy drive.mount(\'/content/drive\') trước.")\n    fid = file_id(raw)\n    state = state if state is not None else {}\n    files_map = state.setdefault("files", {})\n    if fid in files_map and files_map[fid].is_file():\n        return files_map[fid]\n    for cand in (mount / ".shortcut-targets-by-id" / fid,\n                 mount / "MyDrive" / "_v4_shortcuts" / fid):\n        if cand.is_file():\n            files_map[fid] = cand\n            return cand\n    # ponytail: never os.walk entire MyDrive over FUSE (hangs on thousands of dirs); scan only explicit data_dir\n    if not state.get("indexed") and state.get("wanted_ids") and str(state.get("data_dir", "")).strip():\n        files_map.update(index_mounted_files(\n            state["wanted_ids"], state["data_dir"], state.get("exclude_dir")))\n        state["indexed"] = True\n        if fid in files_map and files_map[fid].is_file():\n            return files_map[fid]\n    if drive is None:\n        by_id = mount / ".shortcut-targets-by-id" / fid\n        if by_id.is_file():\n            files_map[fid] = by_id\n            return by_id\n        raise FileNotFoundError(f"Không tìm thấy Drive ID {fid} trên {mount} (thiếu Drive metadata client).")\n\n    # ponytail: metadata-only parent walk (0 bytes downloaded -> never hits Drive download quota)\n    meta_cache = state.setdefault("meta", {})\n    folders = state.setdefault("folders", {})\n    if "root_id" not in state:\n        state["root_id"] = drive.files().get(fileId="root", fields="id").execute(num_retries=5)["id"]\n        folders[state["root_id"]] = mount / "MyDrive"\n\n    def get_meta(item_id):\n        if item_id not in meta_cache:\n            meta_cache[item_id] = drive.files().get(\n                fileId=item_id,\n                fields="id,name,mimeType,parents,driveId,shortcutDetails,owners",\n                supportsAllDrives=True,\n            ).execute(num_retries=5)\n        return meta_cache[item_id]\n\n    chain, seen, cur, base = [], set(), fid, None\n    top_folder_id, top_folder_name, leaf_meta, leaf_err = None, None, None, None\n    while cur and cur not in seen:\n        seen.add(cur)\n        if cur in folders and folders[cur].is_dir():\n            base = folders[cur]\n            break\n        if chain:\n            sc_dir = mount / ".shortcut-targets-by-id" / cur\n            if sc_dir.is_dir():\n                base = folders[cur] = sc_dir\n                break\n        try:\n            meta = get_meta(cur)\n        except Exception as exc:\n            if cur == fid:\n                leaf_err = exc\n            break\n        if cur == fid:\n            leaf_meta = meta\n        target_id = (meta.get("shortcutDetails") or {}).get("targetId")\n        if target_id:\n            cur = target_id\n            continue\n        if meta.get("mimeType") == "application/vnd.google-apps.folder":\n            top_folder_id, top_folder_name = cur, meta.get("name")\n        parents = meta.get("parents") or []\n        if not parents:\n            drive_id = meta.get("driveId")\n            if cur == state["root_id"]:\n                base = mount / "MyDrive"\n            elif drive_id and cur == drive_id:\n                sd_root = mount / "Shareddrives"\n                cand = sd_root / meta["name"]\n                if cand.is_dir():\n                    base = cand\n                elif sd_root.is_dir():\n                    base = next((p for p in sd_root.iterdir() if _xattr_drive_id(p) == drive_id), None)\n            elif top_folder_id == cur:\n                sc_dir = mount / ".shortcut-targets-by-id" / cur\n                if sc_dir.is_dir():\n                    base = sc_dir\n            if base is not None:\n                folders[cur] = base\n            break\n        chain.append((cur, meta["name"]))\n        parent = parents[0]\n        if top_folder_id is None:\n            top_folder_id = parent\n        if parent == state["root_id"]:\n            base = folders[parent] = mount / "MyDrive"\n            break\n        cur = parent\n\n    if base is not None:\n        curr = base\n        for item_id, item_name in reversed(chain):\n            cand = curr / item_name\n            if cand.exists() and (_xattr_drive_id(cand) in (None, item_id)):\n                curr = cand\n            elif curr.is_dir():\n                matched = next((p for p in curr.iterdir() if _xattr_drive_id(p) == item_id), None)\n                curr = matched if matched is not None else cand\n            else:\n                curr = cand\n            if curr.is_dir():\n                folders[item_id] = curr\n        if curr.is_file():\n            files_map[fid] = curr\n            return curr\n\n    if leaf_err is None and state.get("unmounted_targets") is not None:\n        target_id = top_folder_id or fid\n        state["unmounted_targets"][target_id] = (top_folder_name or fid) if top_folder_id else fid\n    fname = (leaf_meta or {}).get("name", fid)\n    owner = (((leaf_meta or {}).get("owners") or [{}])[0]).get("emailAddress", "chủ sở hữu")\n    if leaf_err is not None:\n        raise FileNotFoundError(\n            f"Không truy cập được file {fid} ({leaf_err}). Hãy kiểm tra quyền truy cập hoặc mount đúng tài khoản Google Drive sở hữu dữ liệu."\n        )\n    if top_folder_id:\n        label = f"\'{top_folder_name}\' " if top_folder_name else ""\n        raise FileNotFoundError(\n            f"File \'{fname}\' ({fid}) chưa có trên Drive đã mount ({mount}). "\n            f"Hãy mở thư mục gốc {label}https://drive.google.com/drive/folders/{top_folder_id} "\n            f"-> bấm \'Thêm lối tắt vào Drive\' (Add shortcut to Drive) vào MyDrive rồi chạy lại."\n        )\n    raise FileNotFoundError(\n        f"File \'{fname}\' ({fid}, chủ sở hữu: {owner}) chưa gắn được vào {mount} dù đã tạo lối tắt. "\n        f"Hãy mount đúng tài khoản Google sở hữu ({owner}) hoặc nhờ {owner} chia sẻ thư mục cha."\n    )\n\n\ndef ensure_shortcuts(drive, target_ids, shortcut_dir_name="_v4_shortcuts"):\n    """Create Drive shortcuts in MyDrive/<shortcut_dir_name> so DriveFS mounts shared folders/files."""\n    if not target_ids:\n        return 0\n    q = f"mimeType=\'application/vnd.google-apps.folder\' and name=\'{shortcut_dir_name}\' and \'root\' in parents and trashed=false"\n    res = drive.files().list(q=q, fields="files(id)", pageSize=1).execute(num_retries=5)\n    if res.get("files"):\n        parent_id = res["files"][0]["id"]\n    else:\n        parent_id = drive.files().create(\n            body={"name": shortcut_dir_name, "mimeType": "application/vnd.google-apps.folder", "parents": ["root"]},\n            fields="id",\n        ).execute(num_retries=5)["id"]\n    have, page_token = set(), None\n    while True:\n        resp = drive.files().list(\n            q=f"\'{parent_id}\' in parents and mimeType=\'application/vnd.google-apps.shortcut\' and trashed=false",\n            fields="nextPageToken,files(shortcutDetails)", pageSize=1000, pageToken=page_token,\n        ).execute(num_retries=5)\n        have.update((f.get("shortcutDetails") or {}).get("targetId") for f in resp.get("files", []))\n        page_token = resp.get("nextPageToken")\n        if not page_token:\n            break\n    missing = [(tid, label) for tid, label in target_ids.items() if tid not in have]\n    for i in range(0, len(missing), 50):\n        batch = drive.new_batch_http_request()\n        for tid, label in missing[i:i + 50]:\n            name = tid if label == tid else f"{label}_{tid[:6]}"\n            batch.add(drive.files().create(\n                body={\n                    "name": name,\n                    "mimeType": "application/vnd.google-apps.shortcut",\n                    "parents": [parent_id],\n                    "shortcutDetails": {"targetId": tid},\n                },\n                fields="id",\n                supportsAllDrives=True,\n            ))\n        batch.execute()\n    return len(missing)\n\n\ndef joints_from_vertices(vertices, regressor):\n    vertices = np.asarray(vertices)\n    if vertices.ndim != 3 or vertices.shape[1:] != (6890, 3):\n        raise ValueError(f"Expected [T,6890,3] vertices, got {vertices.shape}")\n    result = np.einsum("jv,tvc->tjc", regressor, vertices, optimize=True)\n    if not np.isfinite(result).all():\n        raise ValueError("Non-finite joint coordinates")\n    return result.astype(np.float32)\n\n\ndef centered(a, names):\n    return a - a[:, [names.index("left_hip"), names.index("right_hip")]].mean(1)[:, None]\n\n\ndef rotation(source, target):\n    u, _, vt = np.linalg.svd(source.reshape(-1, 3).T @ target.reshape(-1, 3))\n    correction = np.eye(3)\n    correction[-1, -1] = np.linalg.det(u @ vt)\n    return u @ correction @ vt\n\n\ndef pa_errors(source, target):\n    source = np.asarray(source, dtype=np.float32)\n    target = np.asarray(target, dtype=np.float32)\n    source = source - source.mean(1, keepdims=True)\n    target = target - target.mean(1, keepdims=True)\n    u, _, vt = np.linalg.svd(np.einsum("tji,tjk->tik", source, target))\n    correction = np.broadcast_to(np.eye(3, dtype=np.float32), u.shape).copy()\n    correction[:, -1, -1] = np.linalg.det(u @ vt)\n    rot = u @ correction @ vt\n    moved = source @ rot\n    scale = (moved * target).sum((1, 2)) / np.maximum((moved * moved).sum((1, 2)), 1e-12)\n    return np.linalg.norm(moved * scale[:, None, None] - target, axis=-1).mean(1)\n\n\nclass AlignmentError(ValueError):\n    """A camera has insufficient overlap or pose agreement with GT."""\n\n\ndef associate(tracks, gt, names, ratios=(0.5, 1.0, 2.0)):\n    """GT-assisted timeline search and track stitching with a switch penalty."""\n    body = [names.index(n) for n in BODY]\n    keys = sorted(tracks)\n    all_ids = np.unique(np.concatenate([np.asarray(tracks[k][0], dtype=int) for k in keys]))\n    if len(all_ids) < 16:\n        raise AlignmentError("Need >=16 camera frames for alignment")\n    # Precompute PA costs against every GT tick once, then search timelines cheaply.\n    costs = np.full((len(all_ids), len(keys), len(gt)), np.inf, dtype=np.float32)\n    joints_at = {}\n    gt_body = np.asarray(gt[:, body], dtype=np.float32)\n    for ki, key in enumerate(keys):\n        ids, joints = tracks[key]\n        ids = np.asarray(ids, dtype=int)\n        if len(np.unique(ids)) != len(ids) or len(ids) != len(joints):\n            raise ValueError(f"Invalid/duplicate frame ids for track {key}")\n        positions = np.searchsorted(all_ids, ids)\n        for s0 in range(0, len(ids), 128):\n            chunk = joints[s0:s0 + 128, body]\n            source = np.repeat(chunk[:, None], len(gt), axis=1)\n            target = np.broadcast_to(gt_body[None], source.shape)\n            costs[positions[s0:s0 + 128], ki] = pa_errors(\n                source.reshape(-1, len(body), 3), target.reshape(-1, len(body), 3)\n            ).reshape(len(chunk), len(gt))\n        for pos, joint in zip(positions, joints):\n            joints_at[(int(pos), ki)] = joint\n    roots = np.zeros((len(all_ids), len(keys), 3))\n    for (pos, ki), joints in joints_at.items():\n        roots[pos, ki] = joints[[names.index("left_hip"), names.index("right_hip")]].mean(0)\n    continuity = {}\n    candidates = []\n    best_lb = float("inf")\n    for ratio in ratios:\n        mapped = np.rint((all_ids - all_ids.min()) * ratio).astype(int)\n        # ponytail: cap required camera frames by achievable GT window when video outlasts mocap clip\n        min_keep = max(16, int(np.ceil(0.7 * min(len(all_ids), np.ceil(len(gt) / ratio)))))\n        for offset in range(-int(mapped.max()), len(gt)):\n            gi = mapped + offset\n            keep = np.where((gi >= 0) & (gi < len(gt)))[0]\n            if len(keep) < min_keep:\n                continue\n            unique_gt, unique = np.unique(gi[keep], return_index=True)\n            positions = keep[unique]\n            coverage = min(1.0, (float(len(unique_gt)) * max(ratio, 1.0)) / len(gt))\n            if len(unique_gt) < 16 or coverage < 0.5:\n                continue\n            local = costs[positions, :, unique_gt]\n            lb = float(local.min(axis=1).mean())\n            best_lb = min(best_lb, lb)\n            if lb > 0.15:\n                continue\n            previous = local[0].copy()\n            parents = []\n            for step, emission in enumerate(local[1:], 1):\n                link = (int(positions[step - 1]), int(positions[step]))\n                if link not in continuity:\n                    distance = np.linalg.norm(roots[link[0], :, None] - roots[link[1], None, :], axis=-1)\n                    continuity[link] = (1 - np.eye(len(keys))) * (0.025 + 0.1 * np.minimum(distance, 5))\n                options = previous[:, None] + continuity[link]\n                parent = np.argmin(options, axis=0)\n                previous = emission + options[parent, np.arange(len(keys))]\n                parents.append(parent)\n            final = int(np.argmin(previous))\n            if not np.isfinite(previous[final]):\n                continue\n            path = [final]\n            for parent in reversed(parents):\n                path.append(int(parent[path[-1]]))\n            path = np.array(path[::-1])\n            raw_error = float(np.mean(local[np.arange(len(path)), path]))\n            score = float(previous[final] / len(path) + 0.03 * (1 - coverage))\n            candidates.append((score, raw_error, ratio, offset, positions, unique_gt, path, coverage))\n    if not candidates:\n        if np.isfinite(best_lb):\n            raise AlignmentError(f"Alignment PA error {best_lb * 1000:.1f}mm exceeds 150mm; inspect data")\n        raise AlignmentError("No alignment covers >=70% achievable camera overlap and >=50% GT timeline")\n    candidates.sort(key=lambda c: c[:4])\n    score, error, ratio, offset, positions, gi, path, coverage = candidates[0]\n    if error > 0.15:\n        raise AlignmentError(f"Alignment PA error {error * 1000:.1f}mm exceeds 150mm; inspect data")\n    joints = np.array([joints_at[(int(pos), int(ki))] for pos, ki in zip(positions, path)])\n    segments = []\n    for i, ki in enumerate(path):\n        key = keys[ki]\n        if not segments or segments[-1]["track"] != key:\n            segments.append({"track": key, "gt_start": int(gi[i]), "gt_end": int(gi[i])})\n        else:\n            segments[-1]["gt_end"] = int(gi[i])\n    report = dict(track=segments[0]["track"] if len(segments) == 1 else "stitched",\n                  segments=segments, ratio=ratio, offset=offset, matched=len(gi),\n                  coverage=coverage, pa_mm=error * 1000,\n                  switches=len(segments)-1,\n                  runner_up_score_mm=candidates[1][0] * 1000 if len(candidates) > 1 else None)\n    return gi, joints, report\n\n\ndef synchronize(ids_m, jm, ids_s, js):\n    """Interpolate cameras onto shared GT ticks; never bridge missing track gaps."""\n    common = np.arange(max(ids_m.min(), ids_s.min()), min(ids_m.max(), ids_s.max()) + 1)\n    arrays, valid = [], np.ones(len(common), dtype=bool)\n    for ids, joints in ((ids_m, jm), (ids_s, js)):\n        hi = np.minimum(np.searchsorted(ids, common), len(ids) - 1)\n        lo = np.maximum(hi - 1, 0)\n        step = max(1.0, float(np.median(np.diff(ids))))\n        valid &= (common == ids[hi]) | ((ids[hi] - ids[lo]) <= 1.5 * step)\n        flat = joints.reshape(len(ids), -1)\n        values = np.stack([np.interp(common, ids, column) for column in flat.T], axis=1)\n        arrays.append(values.reshape(len(common), *joints.shape[1:]).astype(np.float32))\n    return common[valid], arrays[0][valid], arrays[1][valid]\n\n\ndef _download_drive_file(drive, fid, dest):\n    """Stream a Drive file via OAuth API (or gdown if quota-limited) to a temporary path."""\n    from googleapiclient.http import MediaIoBaseDownload\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    temp = dest.with_suffix(".download")\n    try:\n        with temp.open("wb") as stream:\n            reader = MediaIoBaseDownload(\n                stream,\n                drive.files().get_media(fileId=fid, supportsAllDrives=True),\n                chunksize=32 * 1024 * 1024,\n            )\n            done = False\n            while not done:\n                _, done = reader.next_chunk(num_retries=5)\n        temp.replace(dest)\n        return dest\n    except Exception:\n        temp.unlink(missing_ok=True)\n        import gdown\n        out = gdown.download(id=fid, output=str(temp), quiet=True, fuzzy=True)\n        if not out or not temp.is_file() or not temp.stat().st_size:\n            raise\n        temp.replace(dest)\n        return dest\n\n\ndef _resolve_or_download(drive, ref, mount, temp_dir, mount_state):\n    """Prefer mounted Drive (0% download); fall back to OAuth API only for unmounted orphan files."""\n    try:\n        return resolve_mounted(drive, ref, mount, mount_state), False\n    except FileNotFoundError:\n        if drive is None:\n            raise\n        fid = file_id(ref)\n        print(f"  [Fallback API] Đang đọc file chia sẻ lẻ {fid}...", flush=True)\n        return _download_drive_file(drive, fid, Path(temp_dir) / fid), True\n\n\ndef prepare(rows, drive, cache, config, alignment_path=None, skip_unaligned_actions=False,\n            drive_mount="/content/drive", data_dir=""):\n    validate_rows(rows)\n    mapping = load_keypoints3d_map(str(ROOT / "configs/keypoints3D_map.yml"))\n    names = [k["name"] for k in mapping["keypoints"]]\n    reg = np.load(ROOT / "models/J_regressor_body25_extended.npy")\n    reg = reg[[k["regressor_index"] for k in mapping["keypoints"]]]\n    wanted_ids = {file_id(r["GT"]) for r in rows} | {file_id(r["file PKL"]) for r in rows}\n    mount_state = {\n        "wanted_ids": wanted_ids,\n        "data_dir": data_dir,\n        "exclude_dir": str(Path(alignment_path).parent) if alignment_path else None,\n    }\n    temp_dir = Path("/content/v4_tmp") if Path("/content").is_dir() else Path(cache) / "_tmp"\n    pairs, reports = [], []\n    for action in dict.fromkeys(r["action"] for r in rows):\n        cameras = []\n        group = sorted((r for r in rows if r["action"] == action), key=lambda r: int(r["cam"][3:]))\n        if len({file_id(r["GT"]) for r in group}) != 1:\n            raise ValueError(f"Different GT files in one sequence: {action}")\n        gt_fid = file_id(group[0]["GT"])\n        gt_jcache = Path(cache) / f"{gt_fid}.gt_joints.joblib" if drive is not None else None\n        if gt_jcache is not None and gt_jcache.exists():\n            gt = joblib.load(gt_jcache)\n        else:\n            gt_ref = group[0].get("GT_path") or group[0]["GT"]\n            gt_path, gt_is_tmp = _resolve_or_download(drive, gt_ref, drive_mount, temp_dir, mount_state)\n            try:\n                with np.load(gt_path, allow_pickle=False) as data:\n                    gt = joints_from_vertices(data["verts"], reg)\n            finally:\n                if gt_is_tmp:\n                    gt_path.unlink(missing_ok=True)\n            if gt_jcache is not None:\n                gt_jcache.parent.mkdir(parents=True, exist_ok=True)\n                tmp_g = gt_jcache.with_suffix(".tmp")\n                joblib.dump(gt, tmp_g)\n                tmp_g.replace(gt_jcache)\n        for row in group:\n            # ponytail: cache compact 21-joint tracks (~100KB) & drop raw temp downloads immediately\n            jcache = Path(cache) / f"{file_id(row[\'file PKL\'])}.joints.joblib" if (drive is not None or row.get("PKL_path")) else None\n            if jcache is not None and jcache.exists():\n                tracks = joblib.load(jcache)\n            else:\n                pkl_ref = row.get("PKL_path") or row["file PKL"]\n                pkl_path, pkl_is_tmp = _resolve_or_download(drive, pkl_ref, drive_mount, temp_dir, mount_state)\n                try:\n                    raw = joblib.load(pkl_path)\n                    tracks = {}\n                    for key, track in raw.items():\n                        if isinstance(track, dict) and "verts" in track and "frame_ids" in track:\n                            tracks[str(key)] = (np.asarray(track["frame_ids"], dtype=int),\n                                                joints_from_vertices(track["verts"], reg))\n                finally:\n                    if pkl_is_tmp:\n                        pkl_path.unlink(missing_ok=True)\n                if jcache is not None and tracks:\n                    jcache.parent.mkdir(parents=True, exist_ok=True)\n                    tmp_j = jcache.with_suffix(".tmp")\n                    joblib.dump(tracks, tmp_j)\n                    tmp_j.replace(jcache)\n            if not tracks:\n                raise ValueError(f"Expected WHAM tracks with camera-frame verts: {action}/{row[\'cam\']}")\n            print(f"Aligning {action}/{row[\'cam\']}", flush=True)\n            try:\n                gi, joints, report = associate(tracks, gt, names)\n            except AlignmentError as exc:\n                frame_ids = np.unique(np.concatenate([track[0] for track in tracks.values()]))\n                report = dict(status="rejected", reason=str(exc), coverage=None, pa_mm=None,\n                              camera_frames=len(frame_ids), gt_frames=len(gt),\n                              frame_start=int(frame_ids.min()) if len(frame_ids) else None,\n                              frame_end=int(frame_ids.max()) if len(frame_ids) else None)\n            else:\n                report = dict(status="accepted", **report)\n                cameras.append((row["cam"], gi, joints))\n            reports.append(dict(action=action, split=row["split"], cam=row["cam"], **report))\n            print(json.dumps(reports[-1]), flush=True)\n            if alignment_path is not None:\n                atomic_json(Path(alignment_path), reports)\n        # One baseline rotation per sequence/camera, shared by every pair and run.\n        body = [names.index(n) for n in BODY]\n        target_rotations = {\n            cam: rotation(centered(gt[ids], names)[:, body], centered(joints, names)[:, body])\n            for cam, ids, joints in cameras\n        }\n        for (cam_m, ids_m, jm), (cam_s, ids_s, js) in itertools.combinations(cameras, 2):\n            common, synced_m, synced_s = synchronize(ids_m, jm, ids_s, js)\n            # Temporal features/stability must never bridge missing GT ticks.\n            stride = float(np.median(np.diff(common))) if len(common) > 1 else 1\n            runs = np.split(np.arange(len(common)), np.where(np.diff(common) != stride)[0] + 1)\n            for run in runs:\n                if len(run) < 8:\n                    continue\n                for cm, cs, master, slave in ((cam_m, cam_s, synced_m[run], synced_s[run]),\n                                              (cam_s, cam_m, synced_s[run], synced_m[run])):\n                    target = centered(gt[common[run]], names)\n                    target = target @ target_rotations[cm]\n                    M = [dict(zip(names, f)) for f in master]\n                    S = [dict(zip(names, f)) for f in slave]\n                    feats, children = extract_pair_features(M, S)\n                    directions = slave_directions(M, S, config.get("rotation_half_window", 4))\n                    pairs.append(dict(action=action, split=group[0]["split"], cameras=[cm, cs],\n                                      names=names, M=M, S=S, features=feats, children=children,\n                                      directions=directions, target=target.astype(np.float32)))\n        if not any(p["action"] == action for p in pairs):\n            reason = (f"{action}: need at least two aligned cameras; got {len(cameras)}."\n                      if len(cameras) < 2 else f"No synchronized segments for {action}.")\n            if not skip_unaligned_actions:\n                raise ValueError(reason + " Inspect rejected cameras in alignment.json and correct their data/timeline.")\n            reports.append(dict(action=action, split=group[0]["split"], cam=None,\n                                status="skipped_action", reason=reason, coverage=None, pa_mm=None))\n            print(json.dumps(reports[-1]), flush=True)\n            if alignment_path is not None:\n                atomic_json(Path(alignment_path), reports)\n            continue\n        print(f"Prepared {action}: {len(cameras)} cameras", flush=True)\n    if {p["split"] for p in pairs} != {"train", "val"}:\n        raise ValueError("No usable pairs for one of the splits; inspect alignment/coverage")\n    return pairs, reports\n\n\ndef export_model(weights, std, active_features=None):\n    model = dict(type="linear", feature_names=list(FEATURE_NAMES), mean=[0.0] * len(std),\n                 std=np.asarray(std).tolist(), weights=np.asarray(weights).tolist())\n    if active_features is not None:\n        model["active_features"] = np.asarray(active_features, dtype=bool).tolist()\n    return model\n\n\ndef training_options(config):\n    opts = dict(mpjpe_weight=0.25, prior_l2=0.001, patience=10, min_delta_mm=0.01,\n                naive_regret_weight=1.0, naive_margin_mm=0.0)\n    opts.update(config.get("training", {}))\n    for key in ("mpjpe_weight", "prior_l2", "min_delta_mm", "naive_regret_weight", "naive_margin_mm"):\n        if not np.isfinite(opts[key]) or opts[key] < 0:\n            raise ValueError(f"training.{key} must be finite and non-negative")\n    if isinstance(opts["patience"], bool) or not isinstance(opts["patience"], int) or opts["patience"] < 1:\n        raise ValueError("training.patience must be a positive integer")\n    return opts\n\n\ndef training_config(pipeline):\n    fusion = pipeline["fusion"]\n    config = copy.deepcopy(fusion["partwise"])\n    config.pop("weight_model", None)\n    naive = copy.deepcopy(fusion.get("naive", {}))\n    if naive.get("use_aotr_stability", False):\n        naive["sequence_stability"] = copy.deepcopy(fusion["aotr"].get("sequence_stability", {}))\n    config["naive_reference"] = naive\n    return config\n\n\ndef fit_signature(identity, config, lr):\n    return hashlib.sha256((identity + json.dumps(config, sort_keys=True) + str(lr) +\n                           "pa+sequence-mpjpe-naive-regret-nonneg-v3").encode()).hexdigest()\n\n\ndef torch_fuse(pair, weights, std):\n    names = pair["names"]\n    master = torch.as_tensor(np.array([[f[n] for n in names] for f in pair["M"]]), dtype=torch.float32)\n    features = torch.as_tensor(pair["features"])\n    w = torch.sigmoid((features / std) @ weights)\n    result = {n: master[:, i] for i, n in enumerate(names)}\n    for child, parent in _ordered_bones(pair["M"][0]):\n        bi = pair["children"].index(child)\n        direction = torch.tensor(np.array([d.get(child, np.zeros(3)) for d in pair["directions"]]), dtype=torch.float32)\n        m = master[:, names.index(child)] - master[:, names.index(parent)]\n        length = torch.linalg.vector_norm(m, dim=1, keepdim=True)\n        unit = w[:, bi, None] * m / length.clamp_min(1e-9) + (1 - w[:, bi, None]) * direction\n        norm = torch.linalg.vector_norm(unit, dim=1, keepdim=True)\n        unit = torch.where(norm < 1e-9, m / length.clamp_min(1e-9), unit / norm.clamp_min(1e-9))\n        delta = torch.where(length < 1e-9, m, unit * length)\n        result[child] = result[parent] + delta\n    output = torch.stack([result[n] for n in names], dim=1)\n    root = output[:, [names.index("left_hip"), names.index("right_hip")]].mean(1, keepdim=True)\n    return output - root\n\n\ndef torch_pa_loss(pred, target, per_frame=False):\n    # ponytail: Envelope-theorem Procrustes PA-MPJPE avoids global yaw drift in uncalibrated v4 GT & SVD backward NaNs\n    pred_c = pred - pred.mean(1, keepdim=True)\n    tgt_c = target - target.mean(1, keepdim=True)\n    with torch.no_grad():\n        u, _, vt = torch.linalg.svd(torch.einsum("tji,tjk->tik", pred_c, tgt_c))\n        corr = torch.eye(3, dtype=pred.dtype, device=pred.device).expand_as(u).clone()\n        corr[:, -1, -1] = torch.det(u @ vt)\n        rot = u @ corr @ vt\n        moved = pred_c @ rot\n        scale = (moved * tgt_c).sum((1, 2)) / (moved * moved).sum((1, 2)).clamp_min(1e-12)\n    errors = torch.linalg.vector_norm((pred_c @ rot) * scale[:, None, None] - tgt_c, dim=-1).mean(-1)\n    return errors if per_frame else errors.mean()\n\n\ndef naive_relative_loss(pa, mpjpe, naive_pa, naive_mpjpe, opts):\n    margin = opts["naive_margin_mm"] / 1000.0\n    regret = (torch.relu(pa - naive_pa + margin) +\n              opts["mpjpe_weight"] * torch.relu(mpjpe - naive_mpjpe + margin)).mean()\n    loss = (pa + opts["mpjpe_weight"] * mpjpe).mean() + opts["naive_regret_weight"] * regret\n    return loss, regret\n\n\ndef selection_key(metrics, naive, opts):\n    margin = opts["naive_margin_mm"]\n    beats = (metrics["pa_mm"] < naive["pa_mm"] - margin and\n             metrics["mpjpe_mm"] < naive["mpjpe_mm"] - margin)\n    return (not beats, metrics["pa_mm"] + opts["mpjpe_weight"] * metrics["mpjpe_mm"])\n\n\ndef naive_prediction(pair, config):\n    # Match the benchmark\'s naive smoothing settings; never fit a rotation to GT here.\n    fused, _ = fuse_naive(pair["M"], pair["S"], config.get("naive_reference", {"master_weight": 0.5}))\n    return np.array([[f[n] for n in pair["names"]] for f in fused], dtype=np.float32)\n\n\ndef validation(pairs, model, config, method="partwise"):\n    # Equal sequence weight prevents sequences with more camera pairs dominating selection.\n    scores, baselines, mpjpe, mono_mpjpe = {}, {}, {}, {}\n    for p in pairs:\n        cfg = copy.deepcopy(config)\n        if model is None:\n            cfg.pop("weight_model", None)\n        else:\n            cfg["weight_model"] = model\n        names = p["names"]\n        inds = [names.index(n) for n in BODY]\n        if method == "naive":\n            output = naive_prediction(p, config)\n        elif method == "partwise":\n            fused, _ = fuse_partwise(p["M"], p["S"], cfg)\n            output = np.array([[f[n] for n in names] for f in fused], dtype=np.float32)\n        else:\n            raise ValueError(f"Unknown validation method: {method}")\n        master = np.array([[f[n] for n in names] for f in p["M"]], dtype=np.float32)\n        for values, pred in ((scores, output), (baselines, master)):\n            values.setdefault(p["action"], []).append(\n                float(pa_errors(pred[:, inds], p["target"][:, inds]).mean() * 1000))\n        for values, pred in ((mpjpe, output), (mono_mpjpe, master)):\n            errors = np.linalg.norm(centered(pred, names)[:, inds] - p["target"][:, inds], axis=-1)\n            values.setdefault(p["action"], []).append(float(errors.mean() * 1000))\n    return dict(zip(("pa_mm", "mono_pa_mm", "mpjpe_mm", "mono_mpjpe_mm"),\n                    [float(np.mean([np.mean(v) for v in values.values()]))\n                     for values in (scores, baselines, mpjpe, mono_mpjpe)]))\n\n\ndef atomic_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    temp.replace(path)\n\n\ndef save_best(out, config, model):\n    atomic_json(out / "best_weights.json", model)\n    patch = copy.deepcopy(config)\n    patch["weight_model"] = {"enabled": True, **{\n        k: model[k] for k in ("type", "feature_names", "mean", "std", "weights", "active_features")}}\n    (out / "best_partwise.yml").write_text(yaml.safe_dump(\n        {"fusion": {"method": "partwise", "partwise": patch}}, sort_keys=False), encoding="utf-8")\n\n\ndef fit(pairs, config, out, epochs=50, lr=0.01, resume=None, identity=""):\n    opts = training_options(config)\n    if epochs < 1 or not np.isfinite(lr) or lr <= 0:\n        raise ValueError("epochs and lr must be positive and finite")\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    train = [p for p in pairs if p["split"] == "train"]\n    val = [p for p in pairs if p["split"] == "val"]\n    if not train or not val:\n        raise ValueError("Both splits need usable pairs")\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    # Antisymmetry requires zero mean; estimate scale on train only.\n    flat = np.concatenate([p["features"].reshape(-1, len(FEATURE_NAMES)) for p in train])\n    if not np.isfinite(flat).all():\n        raise ValueError("Non-finite training features")\n    active = np.ptp(flat, axis=0) > 1e-6\n    std = torch.tensor(np.where(active, np.maximum(np.sqrt(np.mean(flat ** 2, axis=0)), 1e-6), 1.0))\n    # ponytail: anchor init & L2 prior at 0 (w=0.5 symmetric 50-50) so model only deviates when features help\n    prior = torch.zeros(len(FEATURE_NAMES))\n    weights = torch.nn.Parameter(prior.clone())\n    optimizer = torch.optim.Adam([weights], lr=lr)\n    signature = fit_signature(identity, config, lr)\n    history, best, start, stale, stopping_best = [], float("inf"), 1, 0, (True, float("inf"))\n    if resume:\n        ckpt = torch.load(resume, map_location="cpu", weights_only=False)\n        if ckpt["signature"] != signature or not np.allclose(ckpt["std"], std.numpy()):\n            raise ValueError("Resume dataset/config/lr differs")\n        weights.data.copy_(ckpt["weights"])\n        optimizer.load_state_dict(ckpt["optimizer"])\n        history, best, start = ckpt["history"], ckpt["best_val_mm"], ckpt["epoch"] + 1\n        stale, stopping_best = ckpt["stale_epochs"], ckpt["stopping_best"]\n        random.setstate(ckpt["random_state"])\n        torch.set_rng_state(ckpt["torch_rng"])\n        # Restore the selected model even when resuming into a new output folder.\n        save_best(out, config, ckpt["best_model"])\n        atomic_json(out / "history.json", history)\n    elif list(out.glob("epoch_*.pt")):\n        raise ValueError("Output already has checkpoints; resume or choose a new folder")\n    best_model = ckpt["best_model"] if resume else None\n    equal = validation(val, export_model(np.zeros(len(FEATURE_NAMES)), std.numpy(), active), config)\n    depth = validation(val, None, config)\n    naive = validation(val, None, config, method="naive")\n    if start > epochs or stale >= opts["patience"]:\n        print(json.dumps({"status": "already_completed", "epochs": epochs,\n                          "best_epoch": best_model["epoch"] if best_model else None,\n                          "best_val_objective_mm": best,\n                          "val_beats_naive": best_model["val_beats_naive"]}), flush=True)\n        return history\n    # Fixed train-only references are computed once; no GT-dependent inference routing.\n    naive_errors = {}\n    for p in train:\n        inds = [p["names"].index(n) for n in BODY]\n        output = centered(naive_prediction(p, config), p["names"])[:, inds]\n        target = p["target"][:, inds]\n        naive_errors[id(p)] = (torch.as_tensor(pa_errors(output, target)),\n                               torch.as_tensor(np.linalg.norm(output - target, axis=-1).mean(-1)))\n    actions = sorted({p["action"] for p in train})\n    for epoch in range(start, epochs + 1):\n        order = actions.copy()\n        random.shuffle(order)\n        losses, train_pa, train_mpjpe, train_regret = [], [], [], []\n        for action in order:\n            group = [p for p in train if p["action"] == action]\n            optimizer.zero_grad()\n            action_losses = []\n            action_pa, action_mpjpe, action_regret = [], [], []\n            for p in group:\n                inds = [p["names"].index(n) for n in BODY]\n                target = torch.as_tensor(p["target"][:, inds])\n                pred = torch_fuse(p, weights, std)[:, inds]\n                pa = torch_pa_loss(pred, target, per_frame=True)\n                mpjpe = torch.linalg.vector_norm(pred - target, dim=-1).mean(-1)\n                loss, regret = naive_relative_loss(pa, mpjpe, *naive_errors[id(p)], opts)\n                (loss / len(group)).backward()\n                action_losses.append(float(loss.detach()) * 1000)\n                action_pa.append(float(pa.detach().mean()) * 1000)\n                action_mpjpe.append(float(mpjpe.detach().mean()) * 1000)\n                action_regret.append(float(regret.detach()) * 1000)\n            (opts["prior_l2"] * (weights - prior).square().sum()).backward()\n            optimizer.step()\n            with torch.no_grad():\n                # ponytail: all 8 antisymmetric features are defined >0 = Master better; forbid inverted negative weights\n                weights.clamp_(min=0.0)\n                weights[~torch.as_tensor(active)] = 0.0\n            losses.append(float(np.mean(action_losses)))\n            train_pa.append(float(np.mean(action_pa)))\n            train_mpjpe.append(float(np.mean(action_mpjpe)))\n            train_regret.append(float(np.mean(action_regret)))\n        model = export_model(weights.detach().numpy(), std.numpy(), active)\n        metrics = validation(val, model, config)\n        key = selection_key(metrics, naive, opts)\n        beats_naive, score = not key[0], key[1]\n        if not np.isfinite(score):\n            raise ValueError("Non-finite validation score")\n        improved = best_model is None or key < (not best_model["val_beats_naive"], best)\n        if improved:\n            best, best_model = score, dict(model, epoch=epoch, val_objective_mm=score,\n                                           val_mpjpe_mm=metrics["pa_mm"],\n                                           val_sequence_mpjpe_mm=metrics["mpjpe_mm"],\n                                           equal_val_mpjpe_mm=equal["pa_mm"],\n                                           baseline_val_mpjpe_mm=metrics["mono_pa_mm"],\n                                           naive_val_mpjpe_mm=naive["pa_mm"],\n                                           naive_val_sequence_mpjpe_mm=naive["mpjpe_mm"],\n                                           val_pa_gain_vs_naive_mm=naive["pa_mm"] - metrics["pa_mm"],\n                                           val_mpjpe_gain_vs_naive_mm=naive["mpjpe_mm"] - metrics["mpjpe_mm"],\n                                           val_beats_naive=beats_naive)\n        if key[0] < stopping_best[0] or (key[0] == stopping_best[0] and score < stopping_best[1] - opts["min_delta_mm"]):\n            stopping_best, stale = key, 0\n        else:\n            stale += 1\n        history.append(dict(epoch=epoch, train_mpjpe_mm=float(np.mean(train_pa)),\n                            train_sequence_mpjpe_mm=float(np.mean(train_mpjpe)),\n                            train_naive_regret_mm=float(np.mean(train_regret)),\n                            train_objective_mm=float(np.mean(losses)), val_objective_mm=score,\n                            val_mpjpe_mm=metrics["pa_mm"], val_sequence_mpjpe_mm=metrics["mpjpe_mm"],\n                            equal_val_mpjpe_mm=equal["pa_mm"], equal_val_sequence_mpjpe_mm=equal["mpjpe_mm"],\n                            depth_val_mpjpe_mm=depth["pa_mm"], depth_val_sequence_mpjpe_mm=depth["mpjpe_mm"],\n                            naive_val_mpjpe_mm=naive["pa_mm"], naive_val_sequence_mpjpe_mm=naive["mpjpe_mm"],\n                            val_pa_gain_vs_naive_mm=naive["pa_mm"] - metrics["pa_mm"],\n                            val_mpjpe_gain_vs_naive_mm=naive["mpjpe_mm"] - metrics["mpjpe_mm"],\n                            val_beats_naive=beats_naive, best_val_beats_naive=best_model["val_beats_naive"],\n                            baseline_val_mpjpe_mm=metrics["mono_pa_mm"],\n                            baseline_val_sequence_mpjpe_mm=metrics["mono_mpjpe_mm"],\n                            stale_epochs=stale, best_epoch=best_model["epoch"],\n                            best_val_mpjpe_mm=best_model["val_mpjpe_mm"], best_val_objective_mm=best))\n        checkpoint = dict(epoch=epoch, weights=weights.detach().clone(), std=std.numpy(),\n                          optimizer=optimizer.state_dict(), signature=signature, history=history,\n                          best_val_mm=best, best_model=best_model,\n                          stale_epochs=stale, stopping_best=stopping_best,\n                          random_state=random.getstate(), torch_rng=torch.get_rng_state())\n        temp = out / "checkpoint.tmp"\n        torch.save(checkpoint, temp)\n        temp.replace(out / f"epoch_{epoch:04d}.pt")\n        atomic_json(out / f"epoch_{epoch:04d}_weights.json", model)\n        atomic_json(out / "history.json", history)\n        atomic_json(out / "last.json", {"checkpoint": f"epoch_{epoch:04d}.pt", "signature": signature, "epoch": epoch})\n        save_best(out, config, best_model)\n        print(json.dumps(history[-1]), flush=True)\n        if stale >= opts["patience"]:\n            print(json.dumps({"status": "early_stopped", "epoch": epoch, "best_epoch": best_model["epoch"],\n                              "val_beats_naive": best_model["val_beats_naive"]}), flush=True)\n            break\n    print(json.dumps({"status": "finished", "best_epoch": best_model["epoch"],\n                      "val_beats_naive": best_model["val_beats_naive"],\n                      "note": "Validation gate only; independent coco17 benchmark still required."}), flush=True)\n    return history\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--manifest", required=True)\n    parser.add_argument("--drive-mount", default="/content/drive",\n                        help="Root directory where Google Drive is mounted (0 media downloads)")\n    parser.add_argument("--data-dir", default="",\n                        help="Optional dataset folder inside mounted Drive to scan first")\n    parser.add_argument("--cache", default="inputs/v4_cache")\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--lr", type=float, default=0.01)\n    parser.add_argument("--resume")\n    parser.add_argument("--prepare-only", action="store_true")\n    parser.add_argument("--skip-unaligned-actions", action="store_true",\n                        help="Exclude sequences without usable camera pairs and record them in alignment.json")\n    args = parser.parse_args()\n    if args.epochs < 1 or args.lr <= 0:\n        parser.error("epochs and lr must be positive")\n    rows = json.loads(Path(args.manifest).read_text(encoding="utf-8"))\n    validate_rows(rows)\n    config = training_config(yaml.safe_load((ROOT / "configs/pipeline.yml").read_text(encoding="utf-8")))\n    # Validation uses inference guards/stability; optimization remains a differentiable surrogate.\n    need_drive = not all(r.get("GT_path") and r.get("PKL_path") for r in rows)\n    drive = None\n    if need_drive:\n        from google.auth import default\n        from googleapiclient.discovery import build\n        credentials, _ = default(scopes=["https://www.googleapis.com/auth/drive"])\n        drive = build("drive", "v3", credentials=credentials, cache_discovery=False)\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    cache = Path(args.cache)\n    stable_rows = [{k: v for k, v in r.items() if k not in ("GT_path", "PKL_path")} for r in rows]\n    identity = hashlib.sha256(json.dumps(stable_rows, sort_keys=True, ensure_ascii=False).encode() +\n        (ROOT / "models/J_regressor_body25_extended.npy").read_bytes() +\n        Path(__file__).read_bytes() + (ROOT / "fusion_pipeline/partwise.py").read_bytes() +\n        (ROOT / "fusion_pipeline/naive.py").read_bytes() + (ROOT / "fusion_pipeline/aotr_stability.py").read_bytes() +\n        (ROOT / "configs/keypoints3D_map.yml").read_bytes() + json.dumps(config, sort_keys=True).encode() +\n        str(args.skip_unaligned_actions).encode()).hexdigest()\n    prepared_out = out / f"prepared_{identity}.joblib"\n    prepared_cache = cache / f"prepared_{identity}.joblib"\n    if prepared_out.exists():\n        pairs, reports = joblib.load(prepared_out)\n    elif prepared_cache.exists():\n        pairs, reports = joblib.load(prepared_cache)\n    else:\n        pairs, reports = prepare(rows, drive, cache, config, out / "alignment.json",\n                                 skip_unaligned_actions=args.skip_unaligned_actions,\n                                 drive_mount=args.drive_mount, data_dir=args.data_dir)\n        temp = prepared_out.with_suffix(".tmp")\n        joblib.dump((pairs, reports), temp)\n        temp.replace(prepared_out)\n    atomic_json(out / "alignment.json", reports)\n    atomic_json(out / "manifest.json", rows)\n    atomic_json(out / "protocol.json", {\n        "metric": "12-body-joint PA-MPJPE and baseline-sequence-aligned MPJPE (mm), not coco17",\n        "selection": "Prefer checkpoints beating naive on BOTH PA and MPJPE by naive_margin_mm; then minimize PA + mpjpe_weight * MPJPE. Equal segment/action weight.",\n        "loss": "PA + mpjpe_weight * MPJPE + naive_regret_weight * mean(relu(PA_frame - naive_PA_frame + margin) + mpjpe_weight * relu(MPJPE_frame - naive_MPJPE_frame + margin)) + zero-prior L2 (non-negative weights); train references only",\n        "naive_reference": config["naive_reference"],\n        "training": training_options(config),\n        "target_alignment": "one mono-derived rotation per action/camera on all associated frames (12 body joints)",\n        "surrogate": "gradient step excludes non-differentiable inference guards/stability; validation includes them",\n        "association": "GT-assisted proper-rotation PA timeline search and penalized track stitching; ratio 0.5/1/2, integer offset",\n        "limitations": "GT-assisted time/track association; review alignment.json. No independent test score.",\n        "feature_names": list(FEATURE_NAMES), "config": config,\n        "train_actions": sorted({p["action"] for p in pairs if p["split"] == "train"}),\n        "val_actions": sorted({p["action"] for p in pairs if p["split"] == "val"}),\n        "skip_unaligned_actions": args.skip_unaligned_actions,\n        "skipped_actions": [r["action"] for r in reports if r.get("status") == "skipped_action"],\n        "synchronization": "linear interpolation to shared GT ticks, excluding missing-track gaps",\n        "directed_segments": len(pairs),\n        "split_counts": {split: {\n            "actions": len({p["action"] for p in pairs if p["split"] == split}),\n            "directed_camera_pairs": len({(p["action"], *p["cameras"]) for p in pairs if p["split"] == split}),\n            "directed_segments": sum(p["split"] == split for p in pairs),\n            "segment_frames": sum(len(p["M"]) for p in pairs if p["split"] == split),\n        } for split in ("train", "val")},\n    })\n    if args.prepare_only:\n        print("Prepared data and alignment report:", out / "alignment.json")\n        return\n    fit(pairs, config, out, args.epochs, args.lr, args.resume, identity)\n\n\nif __name__ == "__main__":\n    main()\n', 'fusion_pipeline/partwise.py': '"""Part-wise symmetric two-view fusion (calibration-free, GT-free).\n\nFor every bone, the Master and Slave directions are blended and the skeleton is rebuilt\nby forward kinematics with the Master bone lengths (so MBLE equals the Master/baseline):\n\n    u = normalize(w * u_M + (1 - w) * R_part @ u_S)\n\n* R_part maps the Slave\'s torso part onto the Master\'s, with no extrinsics:\n  R_part = B_M @ B_S^T, where B is an orthonormal body frame built from\n  - lower part: hip axis + spine (pelvis->neck), used for hips/legs/feet;\n  - upper part: shoulder axis + spine, used for head/shoulders/arms/hands.\n  A single whole-body rotation twists the upper body when one view is poor; anchoring\n  each limb on the torso part it hangs from avoids that. R_part is smoothed over time.\n* w is the Master weight per frame: a logistic function of the root-depth difference\n  (depth_M - depth_S, camera frame). The farther view tends to be the more accurate\n  one on the design split (frame AUC 0.825; 0.863 on held-out). The logistic parameters\n  were fitted once on the design split and are frozen in the config.\n* A sequence-stability pass (fusion_pipeline.aotr_stability) then shrinks the\n  correction residual F - M where it would add acceleration.\n\nPelvis and neck stay at the Master positions. Only camera1 (Master) is modified.\n"""\nimport json\nfrom pathlib import Path\nimport numpy as np\n\nfrom fusion_pipeline.adaptive_naive import bone_confidence, bone_jitter\nfrom fusion_pipeline.aotr import _ordered_bones\nfrom fusion_pipeline.detector import compute_bone_distortion_score\n\nLOWER_PART = frozenset({\n    "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle",\n    "left_foot", "right_foot", "left_toe", "right_toe",\n})\nREQUIRED = ("pelvis", "neck", "left_hip", "right_hip", "left_shoulder", "right_shoulder")\n\n\ndef _as_pose(frame):\n    return {k: np.asarray(v, dtype=float) for k, v in frame.items()}\n\n\ndef _orthonormalize(R):\n    U, _, Vt = np.linalg.svd(R)\n    D = np.eye(3)\n    D[2, 2] = np.sign(np.linalg.det(U @ Vt))\n    return U @ D @ Vt\n\n\ndef part_frame(pose, left, right):\n    """Orthonormal frame (columns x, y, z): x = right->left joint, y = spine, z = x × y."""\n    x = pose[left] - pose[right]\n    x = x / np.linalg.norm(x)\n    y = pose["neck"] - pose["pelvis"]\n    y = y - x * (x @ y)\n    y = y / np.linalg.norm(y)\n    return np.stack([x, y, np.cross(x, y)], axis=1)\n\n\ndef root_depth(pose):\n    return float((0.5 * (pose["left_hip"] + pose["right_hip"]))[2])\n\n\nLOWER_TORSO_JOINTS = ("pelvis", "neck", "left_hip", "right_hip")\nUPPER_TORSO_JOINTS = ("pelvis", "neck", "left_shoulder", "right_shoulder")\n\n\ndef _depth_logits(master, slave, logit):\n    d = np.array([root_depth(m) - root_depth(s) for m, s in zip(master, slave)], dtype=float)\n    d = np.nan_to_num(d)\n    return float(logit["a"]) + float(logit["b"]) * (d - float(logit["mu"])) / float(logit["sd"])\n\n\ndef _sigmoid(z):\n    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))\n\n\ndef master_weights(master, slave, logit):\n    """Per-frame Master weight from the root-depth difference (farther view favoured)."""\n    return _sigmoid(_depth_logits(master, slave, logit))\n\n\ndef _part_vis_weight(vis_m, vis_s, joints):\n    if not vis_m and not vis_s:\n        return 1.0\n    vm = vis_m or {}\n    vs = vis_s or {}\n    both = sum(bool(vm.get(j, True)) and bool(vs.get(j, True)) for j in joints)\n    return float(both) / float(len(joints))\n\n\nPRIMARY_LIMB_JOINTS = frozenset({\n    "left_elbow", "right_elbow", "left_wrist", "right_wrist",\n    "left_knee", "right_knee", "left_ankle", "right_ankle",\n})\n\n\ndef _median_bone_disagreement_deg(master_pose, slave_dirs, vis_m=None, vis_s=None):\n    """Zero-GT cross-view pose disagreement: median angle (deg) across primary limb bones."""\n    vm, vs = vis_m or {}, vis_s or {}\n    vis_angles, all_angles = [], []\n    for child, parent in _ordered_bones(master_pose):\n        if child not in PRIMARY_LIMB_JOINTS:\n            continue\n        us = slave_dirs.get(child)\n        if us is None:\n            continue\n        vec = master_pose[child] - master_pose[parent]\n        n = float(np.linalg.norm(vec))\n        if n <= 1e-9:\n            continue\n        ang = float(np.degrees(np.arccos(np.clip(float((vec / n) @ us), -1.0, 1.0))))\n        all_angles.append(ang)\n        if vm.get(child, True) and vs.get(child, True):\n            vis_angles.append(ang)\n    sample = vis_angles if len(vis_angles) >= 3 else all_angles\n    return float(np.median(sample)) if sample else 0.0\n\n\ndef _frame_bone_jump_deg(prev_pose, curr_pose):\n    if prev_pose is None:\n        return 0.0\n    angles = []\n    for child, parent in _ordered_bones(curr_pose):\n        v0 = prev_pose[child] - prev_pose[parent]\n        v1 = curr_pose[child] - curr_pose[parent]\n        n0, n1 = float(np.linalg.norm(v0)), float(np.linalg.norm(v1))\n        if n0 > 1e-9 and n1 > 1e-9:\n            angles.append(float(np.degrees(np.arccos(np.clip(float((v0 / n0) @ (v1 / n1)), -1.0, 1.0)))))\n    return float(np.median(angles)) if angles else 0.0\n\n\ndef slave_directions(master, slave, half_window=4, vis_master=None, vis_slave=None, frame_weights=None):\n    """Slave bone unit vectors in the Master frame, one dict per frame."""\n    n_frames = len(master)\n    rot = {"lower": [], "upper": []}\n    part_w = {"lower": np.ones(n_frames, dtype=float), "upper": np.ones(n_frames, dtype=float)}\n    for t, (m, s) in enumerate(zip(master, slave)):\n        rot["lower"].append(part_frame(m, "left_hip", "right_hip") @ part_frame(s, "left_hip", "right_hip").T)\n        rot["upper"].append(part_frame(m, "left_shoulder", "right_shoulder")\n                            @ part_frame(s, "left_shoulder", "right_shoulder").T)\n        base_fw = 1.0 if frame_weights is None else float(frame_weights[t])\n        vm = vis_master[t] if vis_master is not None else None\n        vs = vis_slave[t] if vis_slave is not None else None\n        part_w["lower"][t] = base_fw * _part_vis_weight(vm, vs, LOWER_TORSO_JOINTS)\n        part_w["upper"][t] = base_fw * _part_vis_weight(vm, vs, UPPER_TORSO_JOINTS)\n\n    # ponytail: precompute nearest clean frame fallback when an entire local window is occluded\n    clean_idx = {k: np.where(part_w[k] > 0.5)[0] for k in ("lower", "upper")}\n    bones = _ordered_bones(master[0])\n    out = []\n    for t, s in enumerate(slave):\n        lo, hi = max(0, t - half_window), min(n_frames, t + half_window + 1)\n        R = {}\n        for k in ("lower", "upper"):\n            weights = part_w[k][lo:hi]\n            w_sum = float(np.sum(weights))\n            if w_sum > 1e-6:\n                R_avg = sum(w_i * R_i for w_i, R_i in zip(weights, rot[k][lo:hi])) / w_sum\n                R[k] = _orthonormalize(R_avg)\n            elif len(clean_idx[k]) > 0:\n                nearest = int(clean_idx[k][np.argmin(np.abs(clean_idx[k] - t))])\n                R[k] = _orthonormalize(rot[k][nearest])\n            else:\n                R[k] = _orthonormalize(np.mean(rot[k][lo:hi], axis=0))\n        dirs = {}\n        for child, parent in bones:\n            v = s[child] - s[parent]\n            n = np.linalg.norm(v)\n            if n > 1e-9:\n                dirs[child] = R["lower" if child in LOWER_PART else "upper"] @ (v / n)\n        out.append(dirs)\n    return out\n\n\ndef blend_frame(master_pose, slave_dirs, w, bone_weights=None):\n    """Blend bone directions (w = Master weight, or per-bone dict) and rebuild with Master bone lengths."""\n    out = {k: v.copy() for k, v in master_pose.items()}\n    for child, parent in _ordered_bones(master_pose):\n        um = master_pose[child] - master_pose[parent]\n        length = np.linalg.norm(um)\n        if length < 1e-9:\n            out[child] = out[parent] + (master_pose[child] - master_pose[parent])\n            continue\n        um = um / length\n        us = slave_dirs.get(child)\n        wb = float(bone_weights[child]) if (bone_weights is not None and child in bone_weights) else float(w)\n        u = um if us is None else wb * um + (1.0 - wb) * us\n        nu = np.linalg.norm(u)\n        u = um if nu < 1e-9 else u / nu\n        out[child] = out[parent] + length * u\n    return out\n\n\ndef _geodesic_deg(R1, R2):\n    tr = float(np.trace(R1.T @ R2))\n    return float(np.degrees(np.arccos(np.clip((tr - 1.0) / 2.0, -1.0, 1.0))))\n\n\ndef _torso_frame(pose):\n    return _orthonormalize(0.5 * (\n        part_frame(pose, "left_hip", "right_hip")\n        + part_frame(pose, "left_shoulder", "right_shoulder")\n    ))\n\n\ndef estimate_static_camera_rotation(master, slave, vis_master=None, vis_slave=None,\n                                    frame_valid=None, inlier_deg=25.0):\n    """Zero-GT robust SO(3) medoid + inlier mean for ONE static camera rotation R_cam (Slave -> Master)."""\n    cands, weights = [], []\n    for t, (m, s) in enumerate(zip(master, slave)):\n        if frame_valid is not None and frame_valid[t] <= 0.0:\n            continue\n        vm = vis_master[t] if vis_master is not None else None\n        vs = vis_slave[t] if vis_slave is not None else None\n        wl = _part_vis_weight(vm, vs, LOWER_TORSO_JOINTS)\n        wu = _part_vis_weight(vm, vs, UPPER_TORSO_JOINTS)\n        if wl >= 0.5:\n            cands.append(part_frame(m, "left_hip", "right_hip") @ part_frame(s, "left_hip", "right_hip").T)\n            weights.append(wl)\n        if wu >= 0.5:\n            cands.append(part_frame(m, "left_shoulder", "right_shoulder") @ part_frame(s, "left_shoulder", "right_shoulder").T)\n            weights.append(wu)\n    if not cands:\n        for m, s in zip(master, slave):\n            cands.append(_torso_frame(m) @ _torso_frame(s).T)\n            weights.append(1.0)\n    step = max(1, len(cands) // 60)\n    best_R, best_med = cands[0], float("inf")\n    for idx in range(0, len(cands), step):\n        R0 = cands[idx]\n        med = float(np.median([_geodesic_deg(R0, R) for R in cands]))\n        if med < best_med:\n            best_med, best_R = med, R0\n    inliers = [(w, R) for w, R in zip(weights, cands) if _geodesic_deg(best_R, R) <= float(inlier_deg)]\n    if not inliers:\n        return _orthonormalize(best_R), best_med\n    w_sum = sum(w for w, _ in inliers)\n    return _orthonormalize(sum(w * R for w, R in inliers) / max(w_sum, 1e-9)), best_med\n\n\ndef _apply_static_global_orientation(fused_lists, M, S, z_depth, vis_master=None, vis_slave=None,\n                                     frame_valid=None, smooth_hw=2):\n    """Decoupled global-orientation fusion for static cameras.\n\n    Applies ONE rigid rotation dR(t) per frame about pelvis so PA-MPJPE and MBLE are\n    mathematically invariant while sequence-aligned MPJPE gains from static R_cam.\n    """\n    R_cam, med_deg = estimate_static_camera_rotation(M, S, vis_master, vis_slave, frame_valid=frame_valid)\n    # ponytail: fail-closed when cross-view R_cam is noisy (e.g. broken Master on MMA/grappling)\n    if med_deg > 10.0:\n        return fused_lists, med_deg\n    bm = [_torso_frame(m) for m in M]\n    bs = [R_cam @ _torso_frame(s) for s in S]\n    torso_j = ("pelvis", "neck", "left_hip", "right_hip", "left_shoulder", "right_shoulder")\n\n    targets = []\n    for t, zt in enumerate(z_depth):\n        vm = vis_master[t] if vis_master is not None else {}\n        vs = vis_slave[t] if vis_slave is not None else {}\n        occ_m = sum(not vm.get(j, True) for j in torso_j)\n        occ_s = sum(not vs.get(j, True) for j in torso_j)\n        zt_eff = float(zt) + 0.6 * (occ_s - occ_m)\n        ang = _geodesic_deg(bm[t], bs[t])\n        if ang > 45.0:\n            jm = _geodesic_deg(bm[t - 1], bm[t]) if t > 0 else 0.0\n            js = _geodesic_deg(bs[t - 1], bs[t]) if t > 0 else 0.0\n            if jm > max(15.0, 2.0 * js):\n                zt_eff = min(zt_eff, 0.0) - 3.0\n            elif js > max(15.0, 2.0 * jm):\n                zt_eff = max(zt_eff, 0.0) + 3.0\n        w_ori = float(_sigmoid(zt_eff))\n        if ang > 140.0:\n            B_tgt = bm[t] if w_ori >= 0.5 else bs[t]\n        else:\n            B_tgt = _orthonormalize(w_ori * bm[t] + (1.0 - w_ori) * bs[t])\n        targets.append(B_tgt)\n\n    # Smooth target orientation locally (skip averaging across >45 deg discontinuities)\n    n = len(targets)\n    out = []\n    for t, pose_dict in enumerate(fused_lists):\n        if smooth_hw > 0 and n > 2:\n            lo, hi = max(0, t - smooth_hw), min(n, t + smooth_hw + 1)\n            local = [targets[k] for k in range(lo, hi) if _geodesic_deg(targets[t], targets[k]) <= 45.0]\n            B_smooth = _orthonormalize(np.mean(local, axis=0)) if local else targets[t]\n        else:\n            B_smooth = targets[t]\n        pose_arr = {k: np.asarray(v, dtype=float) for k, v in pose_dict.items()}\n        # Rotate from the fused skeleton\'s ACTUAL torso frame to B_smooth (avoids double-rotating\n        # when windowed R_part already shifted hips/shoulders toward Slave).\n        dR = B_smooth @ _torso_frame(pose_arr).T\n        root = pose_arr["pelvis"]\n        out.append({k: (root + dR @ (v - root)).tolist() for k, v in pose_arr.items()})\n    return out, med_deg\n\n\nFEATURE_NAMES = (\n    "d_depth", "d_occ_joint", "d_occ_torso", "d_conf2d",\n    "d_bone_jit", "d_cam_accel", "d_bone_distort", "d_facing",\n)\n\n\ndef _facing_abs_cos(pose):\n    """|cos(angle)| between torso normal and camera optical ray -> 1 = frontal, 0 = side profile."""\n    x = pose["left_shoulder"] - pose["right_shoulder"]\n    y = pose["neck"] - pose["pelvis"]\n    normal = np.cross(x, y)\n    nn = float(np.linalg.norm(normal))\n    ray = 0.5 * (pose["left_hip"] + pose["right_hip"])\n    rn = float(np.linalg.norm(ray))\n    if nn <= 1e-9 or rn <= 1e-9:\n        return 0.5\n    return float(abs((normal / nn) @ (ray / rn)))\n\n\ndef _seq_median_accel_mm(poses):\n    if len(poses) < 3:\n        return 0.0\n    keys = sorted(poses[0])\n    arr = np.array([[p[k] for k in keys] for p in poses], dtype=float)\n    acc = np.linalg.norm(arr[2:] - 2.0 * arr[1:-1] + arr[:-2], axis=-1)\n    return float(np.median(acc[5:] if len(acc) > 8 else acc) * 1000.0)\n\n\ndef _depth_outlier_flags(z_m_arr, z_s_arr, ratio=1.5, min_m=10.0):\n    # ponytail: shared bbox-collapse depth explosion guard (z >= 10m & >1.5x other/median)\n    z_m_med = float(np.median(z_m_arr[np.isfinite(z_m_arr)])) if np.any(np.isfinite(z_m_arr)) else 5.0\n    z_s_med = float(np.median(z_s_arr[np.isfinite(z_s_arr)])) if np.any(np.isfinite(z_s_arr)) else 5.0\n    s_exp = (z_s_arr >= min_m) & ((z_s_arr > ratio * np.maximum(z_m_arr, 1e-3)) | (z_s_arr > ratio * max(z_s_med, 1e-3)))\n    m_exp = (z_m_arr >= min_m) & ((z_m_arr > ratio * np.maximum(z_s_arr, 1e-3)) | (z_m_arr > ratio * max(z_m_med, 1e-3)))\n    return m_exp & ~s_exp, s_exp & ~m_exp\n\n\ndef extract_pair_features(M, S, vis_master=None, vis_slave=None, conf_master=None, conf_slave=None):\n    """Return (features[T, B, 8], bone_children) with antisymmetric features phi(M, S) = -phi(S, M)."""\n    n = len(M)\n    bones = _ordered_bones(M[0])\n    b_children = [c for c, _ in bones]\n    z_m_arr = np.array([root_depth(m) for m in M], dtype=float)\n    z_s_arr = np.array([root_depth(s) for s in S], dtype=float)\n    d_depth = np.clip(np.nan_to_num(z_m_arr - z_s_arr), -3.0, 3.0)\n    m_exp, s_exp = _depth_outlier_flags(z_m_arr, z_s_arr)\n    d_depth = np.where(s_exp, 3.0, np.where(m_exp, -3.0, d_depth))\n    jm, js = bone_jitter(M, bones, 3), bone_jitter(S, bones, 3)\n    j_med = {}\n    for c in b_children:\n        pooled = np.concatenate([jm[c], js[c]])\n        pooled = pooled[np.isfinite(pooled)]\n        j_med[c] = float(np.median(pooled)) if pooled.size and float(np.median(pooled)) > 1e-12 else 1.0\n    acc_m, acc_s = _seq_median_accel_mm(M), _seq_median_accel_mm(S)\n    d_cam_acc = float(np.clip(np.log((acc_s + 1.0) / (acc_m + 1.0)), -3.0, 3.0))\n\n    feats = np.zeros((n, len(bones), len(FEATURE_NAMES)), dtype=np.float32)\n    for t in range(n):\n        vm = (vis_master[t] if vis_master else None) or {}\n        vs = (vis_slave[t] if vis_slave else None) or {}\n        cm_t = conf_master[t] if conf_master else None\n        cs_t = conf_slave[t] if conf_slave else None\n        occ_m = float(sum(not vm.get(j, True) for j in M[t])) / max(len(M[t]), 1)\n        occ_s = float(sum(not vs.get(j, True) for j in S[t])) / max(len(S[t]), 1)\n        d_occ_torso = occ_s - occ_m\n        d_distort = float(np.clip(compute_bone_distortion_score(S[t]) - compute_bone_distortion_score(M[t]), -1.0, 1.0))\n        d_face = _facing_abs_cos(M[t]) - _facing_abs_cos(S[t])\n        for bi, (child, parent) in enumerate(bones):\n            d_occ_j = float(bool(vm.get(child, True))) - float(bool(vs.get(child, True)))\n            c_m = bone_confidence(cm_t, child, parent, 0.5)\n            c_s = bone_confidence(cs_t, child, parent, 0.5)\n            d_c2d = float(np.log((c_m + 0.05) / (c_s + 0.05)))\n            nj_m = float(jm[child][t] / j_med[child]) if np.isfinite(jm[child][t]) else 0.0\n            nj_s = float(js[child][t] / j_med[child]) if np.isfinite(js[child][t]) else 0.0\n            d_jit = float(np.clip(np.log((nj_s + 0.1) / (nj_m + 0.1)), -3.0, 3.0))\n            feats[t, bi] = (d_depth[t], d_occ_j, d_occ_torso, d_c2d, d_jit, d_cam_acc, d_distort, d_face)\n    return feats, b_children\n\n\ndef eval_weight_model(feats, wm_cfg):\n    """Evaluate learned antisymmetric weight model -> w_Master[T, B] in (0, 1)."""\n    if isinstance(wm_cfg.get("path"), str) and wm_cfg["path"].strip():\n        p = Path(wm_cfg["path"])\n        if not p.is_absolute():\n            p = Path(__file__).resolve().parents[1] / p\n        wm_cfg = {**json.loads(p.read_text(encoding="utf-8")), **wm_cfg}\n    mean = np.asarray(wm_cfg.get("mean", [0.0] * feats.shape[-1]), dtype=float)\n    std = np.maximum(np.asarray(wm_cfg.get("std", [1.0] * feats.shape[-1]), dtype=float), 1e-6)\n    # Center-only or symmetric scale: to preserve phi(M,S) = -phi(S,M), mean must be 0!\n    x = (feats - mean) / std\n    x = np.where(np.asarray(wm_cfg.get("active_features", [True] * feats.shape[-1])), x, 0.0)\n    kind = str(wm_cfg.get("type", "linear")).lower()\n    if kind == "sym_mlp":\n        W1 = np.asarray(wm_cfg["W1"], dtype=float)\n        b1 = np.asarray(wm_cfg["b1"], dtype=float)\n        W2 = np.asarray(wm_cfg["W2"], dtype=float).reshape(-1)\n        logits = 0.5 * (np.tanh(x @ W1 + b1) @ W2 - np.tanh((-x) @ W1 + b1) @ W2)\n    else:\n        w_vec = np.asarray(wm_cfg["weights"], dtype=float).reshape(-1)\n        logits = x @ w_vec\n    return _sigmoid(logits)\n\n\ndef fuse_partwise(master_frames, slave_frames, config, vis_master=None, vis_slave=None,\n                  conf_master=None, conf_slave=None):\n    """Return (camera1 frames as lists, diagnostics). Inputs: per-frame {joint: xyz}."""\n    if len(master_frames) != len(slave_frames) or not master_frames:\n        raise ValueError("partwise fusion needs equally long, non-empty Master/Slave sequences")\n    M = [_as_pose(f) for f in master_frames]\n    S = [_as_pose(f) for f in slave_frames]\n    missing = [j for j in REQUIRED if j not in M[0] or j not in S[0]]\n    if missing:\n        raise ValueError(f"partwise fusion requires joints {missing}")\n    z_m_arr = np.array([root_depth(m) for m in M], dtype=float)\n    z_s_arr = np.array([root_depth(s) for s in S], dtype=float)\n    z_depth = _depth_logits(M, S, config["depth_logit"])\n    w = _sigmoid(z_depth)\n    hw = int(config.get("rotation_half_window", 4))\n    mp_cfg = dict(config.get("multi_person", {}) or {})\n    mp_enabled = bool(mp_cfg.get("enabled", False))\n\n    swap_gated = 0\n    depth_outlier_frames = 0\n    occ_master_bones = 0\n    occ_slave_bones = 0\n    disagreements = []\n\n    wm_cfg = config.get("weight_model")\n    use_wm = isinstance(wm_cfg, dict) and bool(wm_cfg.get("enabled", True))\n    w_mat = None\n    if use_wm:\n        feats, b_children = extract_pair_features(M, S, vis_master, vis_slave, conf_master, conf_slave)\n        w_mat = eval_weight_model(feats, wm_cfg)\n    # ponytail: always guard against occlusion-induced bbox-collapse depth explosion (not gated on mp_enabled)\n    depth_outlier_ratio = float(mp_cfg.get("depth_outlier_ratio", 1.5))\n    depth_outlier_min_m = float(mp_cfg.get("depth_outlier_min_m", 10.0))\n    depth_outlier_bias = float(mp_cfg.get("depth_outlier_logit_bias", 3.0))\n    if mp_enabled:\n        max_depth_diff = float(mp_cfg.get("max_depth_diff_m", 3.0))\n        d_clamped = np.clip(np.nan_to_num(z_m_arr - z_s_arr), -max_depth_diff, max_depth_diff)\n        logit = config["depth_logit"]\n        z_depth = float(logit["a"]) + float(logit["b"]) * (d_clamped - float(logit["mu"])) / float(logit["sd"])\n    m_exp, s_exp = _depth_outlier_flags(z_m_arr, z_s_arr, depth_outlier_ratio, depth_outlier_min_m)\n    for t in range(len(M)):\n        if s_exp[t]:\n            z_depth[t] = max(z_depth[t], 0.0) + depth_outlier_bias\n            depth_outlier_frames += 1\n            if use_wm:\n                w_mat[t, :] = _sigmoid(z_depth[t])\n        elif m_exp[t]:\n            z_depth[t] = min(z_depth[t], 0.0) - depth_outlier_bias\n            depth_outlier_frames += 1\n            if use_wm:\n                w_mat[t, :] = _sigmoid(z_depth[t])\n    w = np.mean(w_mat, axis=1) if use_wm else _sigmoid(z_depth)\n    if mp_enabled:\n        use_vis_rot = bool(mp_cfg.get("visibility_weighted_rotation", True))\n        dirs_init = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n        )\n        anti_swap = bool(mp_cfg.get("anti_swap_gate", True))\n        max_ang = float(mp_cfg.get("max_median_bone_angle_deg", 50.0))\n        frame_valid = np.ones(len(M), dtype=float)\n        frame_gate_keep = [False] * len(M)\n        for t in range(len(M)):\n            vm = vis_master[t] if vis_master is not None else {}\n            vs = vis_slave[t] if vis_slave is not None else {}\n            ang = _median_bone_disagreement_deg(M[t], dirs_init[t], vm, vs)\n            disagreements.append(ang)\n            if anti_swap and ang > max_ang:\n                frame_valid[t] = 0.0\n                jump_m = _frame_bone_jump_deg(M[t - 1] if t > 0 else None, M[t])\n                jump_s = _frame_bone_jump_deg(S[t - 1] if t > 0 else None, S[t])\n                occ_m_ratio = float(sum(not vm.get(j, True) for j in M[t])) / max(len(M[t]), 1)\n                # Keep Master untouched unless Master itself clearly broke/jumped under heavy occlusion\n                if not (occ_m_ratio >= 0.35 and jump_m > max(15.0, 2.0 * jump_s)):\n                    frame_gate_keep[t] = True\n                    swap_gated += 1\n        # Recompute R_part excluding suspected swap frames so they cannot contaminate neighbors\n        dirs = slave_directions(\n            M, S, hw,\n            vis_master=vis_master if use_vis_rot else None,\n            vis_slave=vis_slave if use_vis_rot else None,\n            frame_weights=frame_valid if anti_swap else None,\n        ) if anti_swap and np.any(frame_valid == 0.0) else dirs_init\n\n        m_occ_bias = float(mp_cfg.get("master_occluded_logit_bias", 2.0))\n        s_occ_bias = float(mp_cfg.get("slave_occluded_logit_bias", 2.5))\n        both_occ_bias = float(mp_cfg.get("both_occluded_logit_bias", 0.5))\n        bones = _ordered_bones(M[0])\n        fused = []\n        for t, (m, d, zt, wt) in enumerate(zip(M, dirs, z_depth, w)):\n            if frame_gate_keep[t]:\n                fused.append({k: v.copy() for k, v in m.items()})\n                continue\n            vm = vis_master[t] if vis_master is not None else None\n            vs = vis_slave[t] if vis_slave is not None else None\n            if vm is None and vs is None and not use_wm:\n                fused.append(blend_frame(m, d, wt))\n                continue\n            vm, vs = vm or {}, vs or {}\n            bw = {}\n            for bi, (child, _) in enumerate(bones):\n                base_w = float(w_mat[t, bi]) if use_wm else float(wt)\n                z_bone = float(np.log(np.clip(base_w, 1e-9, 1 - 1e-9) /\n                                      np.clip(1 - base_w, 1e-9, 1))) if use_wm else float(zt)\n                m_vis = bool(vm.get(child, True))\n                s_vis = bool(vs.get(child, True))\n                if not m_vis and s_vis:\n                    # Occluded Master + visible Slave: never let depth prior override visibility\n                    bw[child] = float(_sigmoid(min(z_bone, 0.0) - m_occ_bias))\n                    occ_master_bones += 1\n                elif m_vis and not s_vis:\n                    # Visible Master + occluded Slave: protect Master regardless of depth\n                    bw[child] = float(_sigmoid(max(z_bone, 0.0) + s_occ_bias))\n                    occ_slave_bones += 1\n                elif not m_vis and not s_vis:\n                    bw[child] = float(_sigmoid(z_bone + both_occ_bias))\n                else:\n                    bw[child] = base_w\n            fused.append(blend_frame(m, d, wt, bone_weights=bw))\n    elif use_wm:\n        dirs = slave_directions(M, S, hw)\n        w = np.mean(w_mat, axis=1)\n        fused = [blend_frame(m, d, w[t], bone_weights=dict(zip(b_children, w_mat[t])))\n                 for t, (m, d) in enumerate(zip(M, dirs))]\n    else:\n        dirs = slave_directions(M, S, hw)\n        fused = [blend_frame(m, d, wt) for m, d, wt in zip(M, dirs, w)]\n\n    diag = {\n        "mode": "partwise",\n        "frames": len(fused),\n        "master_weight_mean": float(np.mean(w)),\n        "master_weight_min": float(np.min(w)),\n        "master_weight_max": float(np.max(w)),\n        "weight_model_enabled": use_wm,\n        "multi_person_enabled": mp_enabled,\n        "swap_gated_frames": swap_gated,\n        "depth_outlier_frames": depth_outlier_frames,\n        "occluded_master_bones": occ_master_bones,\n        "occluded_slave_bones": occ_slave_bones,\n        "median_bone_angle_deg_mean": float(np.mean(disagreements)) if disagreements else None,\n        "sequence_stability": {"status": "disabled"},\n    }\n    as_lists = [{k: v.tolist() for k, v in p.items()} for p in fused]\n    stab_cfg = dict(config.get("sequence_stability", {}))\n    if stab_cfg.get("enabled", True):\n        from fusion_pipeline.aotr_stability import postprocess_aotr_3d\n        master_lists = [{k: v.tolist() for k, v in p.items()} for p in M]\n        as_lists, stab_diag = postprocess_aotr_3d(\n            master_lists, as_lists, config=stab_cfg, editable_mask=[True] * len(as_lists))\n        diag["sequence_stability"] = stab_diag\n    if mp_enabled and bool(mp_cfg.get("static_camera_orientation", False)):\n        as_lists, rcam_med = _apply_static_global_orientation(\n            as_lists, M, S, z_depth,\n            vis_master=vis_master, vis_slave=vis_slave,\n            frame_valid=frame_valid if anti_swap else None,\n            smooth_hw=int(mp_cfg.get("orientation_smooth_half_window", 2)),\n        )\n        diag["static_camera_orientation"] = {"enabled": True, "r_cam_median_deg": round(rcam_med, 2)}\n    return as_lists, diag\n', 'fusion_pipeline/naive.py': '"""Naive two-view fusion baseline (fusion.method=naive).\n\nThe simplest calibration-free late fusion, used as the lower bar that any proposed\nfusion has to beat. Per frame, with no learning, no temporal model and no GT:\n\n    1. centre both skeletons at their pelvis;\n    2. rotate the Slave onto the Master with one rigid Kabsch rotation over all shared\n       joints (no scale: both are metric WHAM skeletons);\n    3. average joint positions: F = pelvis_M + w * (M - pelvis_M) + (1 - w) * R (S - pelvis_S).\n\nw is a fixed Master weight (default 0.5 = plain average). Bone lengths are NOT preserved\n(averaging positions shortens bent limbs), which is part of what the baseline shows.\nPelvis stays at the Master position, so root-centred metrics are comparable. Only\ncamera1 (Master) is modified.\n\nWith fusion.naive.use_aotr_stability=true the executor passes AOTR\'s sequence-stability\nblock, and the result goes through the same pass as AOTR (savgol + FK projection onto\nthe Master bone lengths, pelvis and neck at the Master). This removes the smoothing\nadvantage from comparisons with AOTR; bone lengths then equal the Master\'s.\n"""\nimport numpy as np\n\n\ndef kabsch_rotation(src, dst):\n    """Rotation R (3x3, det +1) minimising sum ||R @ src_i - dst_i||^2; rows are points."""\n    H = src.T @ dst\n    U, _, Vt = np.linalg.svd(H)\n    D = np.eye(3)\n    D[2, 2] = np.sign(np.linalg.det(Vt.T @ U.T))\n    return Vt.T @ D @ U.T\n\n\ndef fuse_frame(master, slave, master_weight=0.5):\n    """Fuse one frame. master/slave: {joint: xyz}. Returns {joint: np.ndarray} for Master joints."""\n    M = {k: np.asarray(v, dtype=float) for k, v in master.items()}\n    S = {k: np.asarray(v, dtype=float) for k, v in slave.items()}\n    if "pelvis" not in M or "pelvis" not in S:\n        raise ValueError("naive fusion requires a \'pelvis\' joint in both views")\n    shared = [k for k in M if k in S]\n    if len(shared) < 3:\n        raise ValueError(f"naive fusion needs >= 3 shared joints, got {len(shared)}")\n    cm, cs = M["pelvis"], S["pelvis"]\n    src = np.stack([S[k] - cs for k in shared])\n    dst = np.stack([M[k] - cm for k in shared])\n    R = kabsch_rotation(src, dst)\n    w = float(master_weight)\n    out = {}\n    for k, m in M.items():\n        if k in S:\n            out[k] = cm + w * (m - cm) + (1.0 - w) * (R @ (S[k] - cs))\n        else:\n            out[k] = m.copy()\n    out["pelvis"] = cm.copy()\n    return out\n\n\ndef fuse_naive(master_frames, slave_frames, config=None):\n    """Return (camera1 frames as lists, diagnostics). Inputs: per-frame {joint: xyz}."""\n    config = config or {}\n    if len(master_frames) != len(slave_frames) or not master_frames:\n        raise ValueError("naive fusion needs equally long, non-empty Master/Slave sequences")\n    w = float(config.get("master_weight", 0.5))\n    if not 0.0 <= w <= 1.0:\n        raise ValueError(f"fusion.naive.master_weight must be in [0, 1], got {w}")\n    fused = [fuse_frame(m, s, w) for m, s in zip(master_frames, slave_frames)]\n    diag = {"mode": "naive", "frames": len(fused), "master_weight": w,\n            "sequence_stability": {"status": "disabled"}}\n    as_lists = [{k: v.tolist() for k, v in p.items()} for p in fused]\n    stab_cfg = dict(config.get("sequence_stability") or {})\n    if stab_cfg.get("enabled", False):\n        from fusion_pipeline.aotr_stability import postprocess_aotr_3d\n        master_lists = [{k: [float(x) for x in v] for k, v in m.items()} for m in master_frames]\n        as_lists, stab_diag = postprocess_aotr_3d(\n            master_lists, as_lists, config=stab_cfg, editable_mask=[True] * len(as_lists))\n        diag["sequence_stability"] = stab_diag\n    return as_lists, diag\n', 'fusion_pipeline/aotr_stability.py': '"""AOTR offline stability pass (no GT, no learned component).\n\nWe reduce acceleration of the *correction residual* F-M, not the actual\nmotion. Thus real fast motion in Master is preserved. Joint coordinates are\nreconstructed with forward kinematics after temporal scaling; optional\nanthropometric length calibration is independently configurable.\n\nNot a guarantee for GT acceleration/MBLE/PA-MPJPE. Report each separately.\n"""\nimport numpy as np\nfrom scipy.optimize import minimize\ntry:\n    from scipy.signal import savgol_filter\nexcept ImportError:\n    savgol_filter = None\n\nfrom fusion_pipeline.aotr import _ordered_bones, _unit\n\n\ndef _xyz(frames):\n    if not frames:\n        return None, None\n    names = list(frames[0])\n    if any(set(f) != set(names) for f in frames):\n        raise ValueError("Inconsistent joint sets in AOTR sequence")\n    array = np.array([[frame[j] for j in names] for frame in frames], dtype=float)\n    if array.shape != (len(frames), len(names), 3) or not np.isfinite(array).all():\n        raise ValueError("Invalid AOTR 3D sequence")\n    return names, array\n\n\ndef _residual_acceleration(residual):\n    if len(residual) < 3:\n        return 0.0\n    accel = residual[2:] - 2.0 * residual[1:-1] + residual[:-2]\n    return float(np.sqrt(np.mean(np.sum(accel ** 2, axis=2))))\n\n\ndef _temporal_scales(raw_delta, cfg):\n    """Bounded quadratic minimization over one correction scale per frame.\n\n    For s_t in [s_min, 1], minimize\n       w_shape * sum ||D_t||^2 (s_t-1)^2\n     + w_acc * sum ||Delta2(s_t D_t)||^2\n     + w_vel * sum ||Delta1(s_t D_t)||^2.\n    Frames where D_t == 0 remain zero; no new Slave correction is introduced.\n    """\n    n = len(raw_delta)\n    active = np.sum(raw_delta ** 2, axis=(1, 2)) > 1e-14\n    if n < 3 or not np.any(active):\n        return np.ones(n), {"status": "skipped", "reason": "too_short_or_unchanged"}\n    w_shape = max(1e-9, float(cfg.get("shape_weight", 1.0)))\n    w_acc = max(0.0, float(cfg.get("residual_accel_weight", 10.0)))\n    w_vel = max(0.0, float(cfg.get("residual_velocity_weight", 0.5)))\n    low = float(np.clip(cfg.get("minimum_retention", 0.20), 0.0, 1.0))\n    norm2 = np.sum(raw_delta ** 2, axis=(1, 2))\n\n    def objective(s):\n        residual = s[:, None, None] * raw_delta\n        dev = s - 1.0\n        loss = w_shape * np.dot(norm2, dev * dev)\n        grad = 2.0 * w_shape * norm2 * dev\n        if w_acc:\n            diff = residual[2:] - 2 * residual[1:-1] + residual[:-2]\n            loss += w_acc * float(np.sum(diff * diff))\n            grad[2:] += 2 * w_acc * np.sum(diff * raw_delta[2:], axis=(1, 2))\n            grad[1:-1] -= 4 * w_acc * np.sum(diff * raw_delta[1:-1], axis=(1, 2))\n            grad[:-2] += 2 * w_acc * np.sum(diff * raw_delta[:-2], axis=(1, 2))\n        if w_vel:\n            diff = residual[1:] - residual[:-1]\n            loss += w_vel * float(np.sum(diff * diff))\n            grad[1:] += 2 * w_vel * np.sum(diff * raw_delta[1:], axis=(1, 2))\n            grad[:-1] -= 2 * w_vel * np.sum(diff * raw_delta[:-1], axis=(1, 2))\n        return float(loss), grad\n\n    bounds = [(low, 1.0) if a else (0.0, 0.0) for a in active]\n    x0 = np.where(active, 1.0, 0.0)\n    before = objective(x0)[0]\n    if w_acc == 0.0 and w_vel == 0.0:\n        return x0, {"status": "unchanged", "cost_before": before, "cost_after": before}\n    result = minimize(objective, x0, jac=True, bounds=bounds,\n                      method="L-BFGS-B",\n                      options={"maxiter": int(cfg.get("max_iter", 160)),\n                               "ftol": 1e-11, "gtol": 1e-10, "maxls": 30})\n    s = np.clip(result.x, [b[0] for b in bounds], [b[1] for b in bounds])\n    after = objective(s)[0]\n    if not np.isfinite(after) or after > before + 1e-10:\n        return x0, {"status": "rejected", "cost_before": before,\n                    "cost_after": before}\n    return s, {"status": "optimized" if np.any(s[active] < 0.999999) else "unchanged",\n               "cost_before": before, "cost_after": after,\n               "optimizer_success": bool(result.success)}\n\n\ndef _estimate_master_height(master_frames):\n    """Estimate natural vertical standing height of Master skeleton along kinematic chain."""\n    if not master_frames:\n        return 0.0\n    torso_lens = []\n    for f in master_frames:\n        if "neck" in f and "pelvis" in f:\n            torso_lens.append(float(np.linalg.norm(np.asarray(f["neck"]) - np.asarray(f["pelvis"]))))\n        elif "neck" in f and "left_hip" in f and "right_hip" in f:\n            hip_mid = 0.5 * (np.asarray(f["left_hip"]) + np.asarray(f["right_hip"]))\n            torso_lens.append(float(np.linalg.norm(np.asarray(f["neck"]) - hip_mid)))\n        elif "neck" in f and "left_hip" in f:\n            torso_lens.append(float(np.linalg.norm(np.asarray(f["neck"]) - np.asarray(f["left_hip"]))))\n    med_torso = float(np.median(torso_lens)) if torso_lens else 0.5\n\n    leg_lens = []\n    for f in master_frames:\n        for side in ("left", "right"):\n            h, k, a = f"{side}_hip", f"{side}_knee", f"{side}_ankle"\n            if h in f and k in f and a in f:\n                leg_lens.append(float(np.linalg.norm(np.asarray(f[k]) - np.asarray(f[h])) +\n                                      np.linalg.norm(np.asarray(f[a]) - np.asarray(f[k]))))\n    med_leg = float(np.median(leg_lens)) if leg_lens else 0.8\n\n    head_lens = []\n    for f in master_frames:\n        if "head" in f and "neck" in f:\n            head_lens.append(float(np.linalg.norm(np.asarray(f["head"]) - np.asarray(f["neck"]))))\n        elif "nose" in f and "neck" in f:\n            head_lens.append(float(np.linalg.norm(np.asarray(f["nose"]) - np.asarray(f["neck"]))))\n    med_head = float(np.median(head_lens)) if head_lens else 0.15 * med_torso\n\n    foot_lens = []\n    for f in master_frames:\n        for side in ("left", "right"):\n            a, ft = f"{side}_ankle", f"{side}_foot"\n            if a in f and ft in f:\n                foot_lens.append(float(np.linalg.norm(np.asarray(f[ft]) - np.asarray(f[a]))))\n            elif a in f and f"{side}_toe" in f:\n                foot_lens.append(float(np.linalg.norm(np.asarray(f[f"{side}_toe"]) - np.asarray(f[a]))))\n    med_foot = float(np.median(foot_lens)) if foot_lens else 0.06 * med_leg\n\n    return max(0.5, med_head + med_torso + med_leg + med_foot)\n\n\ndef _anthropometric_targets(master_frames, cfg):\n    """Sequence-stable lengths, inferred from known height or Master\'s limb total.\n\n    Only humerus/forearm and femur/shank are modified. Other Master lengths\n    stay as-is (including shoulders, wrists->hands and toe/foot segments).\n    When arm_upper_fraction or leg_thigh_fraction is omitted, proportions are\n    auto-derived from the subject\'s own Master sequence (generalized, zero-finetuning).\n    """\n    lengths = {}\n    if not cfg.get("enabled", False) or not master_frames:\n        return lengths\n\n    upper_frac = cfg.get("arm_upper_fraction")\n    thigh_frac = cfg.get("leg_thigh_fraction")\n    if upper_frac is not None:\n        upper_frac = float(upper_frac)\n        if not (0.0 < upper_frac < 1.0):\n            raise ValueError("Anatomical length ratios must be in (0,1)")\n    if thigh_frac is not None:\n        thigh_frac = float(thigh_frac)\n        if not (0.0 < thigh_frac < 1.0):\n            raise ValueError("Anatomical length ratios must be in (0,1)")\n\n    known_height = cfg.get("person_height_m")\n    if known_height is not None:\n        try:\n            known_height = float(known_height)\n        except (TypeError, ValueError):\n            known_height = None\n\n    scale = 1.0\n    if known_height is not None and known_height > 0:\n        h_m = _estimate_master_height(master_frames)\n        if h_m > 0:\n            scale = known_height / h_m\n\n    arm_h_frac = cfg.get("arm_height_fraction")\n    leg_h_frac = cfg.get("leg_height_fraction")\n\n    for side in ("left", "right"):\n        arms = (f"{side}_shoulder", f"{side}_elbow", f"{side}_wrist")\n        legs = (f"{side}_hip", f"{side}_knee", f"{side}_ankle")\n        for (root, mid, tip), ratio, h_frac in ((arms, upper_frac, arm_h_frac), (legs, thigh_frac, leg_h_frac)):\n            uppers, lowers = [], []\n            for frame in master_frames:\n                if not all(j in frame for j in (root, mid, tip)):\n                    continue\n                a, b, c = (np.asarray(frame[j], dtype=float) for j in (root, mid, tip))\n                upper, lower = float(np.linalg.norm(b-a)), float(np.linalg.norm(c-b))\n                if upper > 1e-5 and lower > 1e-5:\n                    uppers.append(upper)\n                    lowers.append(lower)\n            if not uppers or not lowers:\n                continue\n\n            med_u = float(np.median(uppers))\n            med_l = float(np.median(lowers))\n            measured_total = med_u + med_l\n\n            if ratio is not None:\n                tgt_u, tgt_l = ratio, 1.0 - ratio\n            else:\n                # Generalized: auto-derived from subject\'s own Master sequence medians\n                tgt_u = med_u / measured_total\n                tgt_l = med_l / measured_total\n\n            if h_frac is not None:\n                total = known_height * float(h_frac)\n            elif known_height is not None and known_height > 0:\n                total = measured_total * scale\n            else:\n                total = measured_total\n\n            lengths[(mid, root)] = total * tgt_u\n            lengths[(tip, mid)] = total * tgt_l\n    return lengths\n\n\ndef _fk_reconstruct(master, fused, scale, lengths):\n    """Interpolate local bone orientation, then project onto prescribed length.\n\n    scale=0 preserves local orientation of Master; scale=1 preserves initial\n    AOTR local orientation. Pelvis and neck stay at the Master positions.\n\n    When child was not corrected by AOTR (fused[child] == master[child]),\n    we use the master bone direction only — blending with uF would be\n    meaningless because uF is not an independent observation, just the\n    master vector shifted by the corrected parent. This avoids spurious\n    accel errors from FK propagation on blind/occluded joints.\n    # ponytail: use_master_dir_if_uncorrected; lift if stability needs downstream blend.\n    """\n    out = {j: np.asarray(p, dtype=float).copy() for j, p in master.items()}\n    for child, parent in _ordered_bones(master):\n        m_child, m_parent = np.asarray(master[child]), np.asarray(master[parent])\n        f_child, f_parent = np.asarray(fused[child]), np.asarray(fused[parent])\n        uM = _unit(m_child - m_parent)\n        if uM is None:\n            continue\n        # Child not directly corrected: use master direction to avoid spurious accel\n        # from parent displacement feeding into the blend direction.\n        if np.linalg.norm(f_child - m_child) <= 1e-7:\n            direction = uM\n        else:\n            uF = _unit(f_child - f_parent)\n            if uF is None:\n                uF = uM\n            direction = _unit((1.0 - scale) * uM + scale * uF)\n            if direction is None:\n                direction = uM\n        target = lengths.get((child, parent), float(np.linalg.norm(m_child-m_parent)))\n        out[child] = out[parent] + target * direction\n    return out\n\n\ndef postprocess_aotr_3d(master_frames, fused_frames, config=None, editable_mask=None):\n    """Return (frames, diagnostics) with no access to GT.\n\n    The optional bone calibration is an *additional hypothesis*, not an\n    AOTR slave transfer. Therefore it is reported separately.\n    """\n    cfg = dict(config or {})\n    temporal = dict(cfg.get("temporal_stability", {}))\n    bone = dict(cfg.get("anatomical_calibration", {}))\n    names, M = _xyz(master_frames)\n    other_names, F = _xyz(fused_frames)\n    if names is None:\n        return [], {"status": "empty"}\n    if names != other_names:\n        raise ValueError("AOTR Master/Fused joint ordering mismatch")\n    editable = (\n        np.ones(len(F), dtype=bool)\n        if editable_mask is None else np.asarray(editable_mask, dtype=bool)\n    )\n    if editable.shape != (len(F),):\n        raise ValueError("editable_mask must have one boolean per frame")\n    if np.any(~editable & (np.sum((F-M)**2, axis=(1,2)) > 1e-14)):\n        raise ValueError("Noneditable AOTR fallback frames contain Slave corrections")\n\n    before = _residual_acceleration(F-M)\n    if temporal.get("enabled", True):\n        scales, optimization = _temporal_scales(F-M, temporal)\n    else:\n        scales = np.ones(len(F))\n        optimization = {"status": "disabled"}\n\n    lengths = _anthropometric_targets(master_frames, bone)\n    if not lengths and (not temporal.get("enabled", True) or\n                        np.allclose(scales, 1.0, atol=1e-7)):\n        return fused_frames, {\n            "status": "unchanged", "temporal": optimization,\n            "initial_residual_accel_m_frame2": before,\n            "final_residual_accel_m_frame2": before,\n            "anatomically_calibrated_bones": 0,\n            "mean_temporal_retention": 1.0,\n        }\n\n    bones = _ordered_bones(master_frames[0])\n    n = len(master_frames)\n    w_len = int(temporal.get("smooth_window", 5))\n    if w_len % 2 == 0:\n        w_len -= 1\n\n    reconstructed_frames = []\n    for i, master in enumerate(master_frames):\n        if not editable[i]:\n            reconstructed_frames.append({j: np.asarray(master[j], dtype=float).copy() for j in names})\n            continue\n        rec = _fk_reconstruct(master, fused_frames[i], scales[i], lengths)\n        for joint in names:\n            if float(np.linalg.norm(rec[joint] - np.asarray(master[joint]))) <= 1e-10:\n                rec[joint] = np.asarray(master[joint]).copy()\n        reconstructed_frames.append(rec)\n\n    # Trajectory smoothing + FK bone projection eliminates camera jitter without flattening motion\n    if savgol_filter is not None and (temporal.get("smooth_trajectories", False) or temporal.get("smooth_directions", False)) and n >= w_len and w_len >= 3:\n        name_to_idx = {name: i for i, name in enumerate(names)}\n        curr_arr = np.array([[reconstructed_frames[t][j] for j in names] for t in range(n)], dtype=float)\n        sm_arr = savgol_filter(curr_arr, window_length=w_len, polyorder=2, axis=0)\n        proj_arr = sm_arr.copy()\n        for t in range(n):\n            if not editable[t]:\n                continue\n            for child, parent in bones:\n                ci, pi = name_to_idx[child], name_to_idx[parent]\n                mc = np.asarray(master_frames[t][child], dtype=float)\n                mp = np.asarray(master_frames[t][parent], dtype=float)\n                tgt = lengths.get((child, parent), float(np.linalg.norm(mc - mp)))\n                diff = sm_arr[t, ci] - proj_arr[t, pi]\n                d_norm = float(np.linalg.norm(diff))\n                u = diff / d_norm if d_norm > 1e-6 else np.array([0.0, -1.0, 0.0])\n                proj_arr[t, ci] = proj_arr[t, pi] + tgt * u\n        for t in range(n):\n            if editable[t]:\n                reconstructed_frames[t] = {names[j]: proj_arr[t, j] for j in range(len(names))}\n\n    output = []\n    for t in range(n):\n        rec = reconstructed_frames[t]\n        for joint in names:\n            if float(np.linalg.norm(rec[joint] - np.asarray(master_frames[t][joint]))) <= 1e-10:\n                rec[joint] = np.asarray(master_frames[t][joint]).copy()\n        output.append({j: rec[j].tolist() for j in names})\n    _, stabilized = _xyz(output)\n    active = np.sum((F-M)**2, axis=(1, 2)) > 1e-14\n    return output, {\n        "status": "applied",\n        "temporal": optimization,\n        "initial_residual_accel_m_frame2": before,\n        "final_residual_accel_m_frame2": _residual_acceleration(stabilized-M),\n        "anatomically_calibrated_bones": len(lengths),\n        "frozen_fallback_frames": int(np.count_nonzero(~editable)),\n        "mean_temporal_retention": float(np.mean(scales[active])) if np.any(active) else 1.0,\n        "minimum_temporal_retention": float(np.min(scales[active])) if np.any(active) else 1.0,\n    }\n', 'tests/test_train_v4.py': 'import copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nimport joblib\nimport numpy as np\nimport torch\n\nfrom train_v4 import (BODY, AlignmentError, associate, centered, export_model, fit,\n                      index_mounted_files, prepare, resolve_mounted, torch_fuse,\n                      validate_rows, synchronize, validation, rotation, training_options,\n                      training_config, naive_relative_loss, selection_key, naive_prediction, torch_pa_loss, pa_errors)\nfrom keypoints_map import load_keypoints3d_map\nfrom fusion_pipeline.partwise import eval_weight_model, extract_pair_features, fuse_partwise, slave_directions\nfrom test_partwise import CFG, skeleton, rotate\n\n\ndef pair(split="train"):\n    master = [skeleton(elbow_lift=0.01 * t) for t in range(10)]\n    slave = [skeleton(depth=6, elbow_lift=0.02 * t) for t in range(10)]\n    names = list(master[0])\n    features, children = extract_pair_features(master, slave)\n    return dict(action=split, split=split, M=master, S=slave, names=names,\n                features=features, children=children,\n                directions=slave_directions(master, slave, 4),\n                target=centered(np.array([[f[n] for n in names] for f in slave]), names).astype(np.float32))\n\n\nclass TrainingTests(unittest.TestCase):\n    def test_prepare_shares_baseline_rotation_across_disconnected_runs(self):\n        names = [k["name"] for k in load_keypoints3d_map("configs/keypoints3D_map.yml")["keypoints"]]\n        ids = np.r_[np.arange(12), np.arange(20, 32)]\n        gt = np.array([[skeleton()[n] for n in names] for _ in range(32)])\n        joints = np.array([[rotate(skeleton(), 0 if t < 12 else 60)[n] for n in names] for t in ids])\n        body = [names.index(n) for n in BODY]\n        expected = centered(gt[ids], names)\n        expected = expected @ rotation(expected[:, body], centered(joints, names)[:, body])\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            np.savez(root / "gt.npz", verts=gt)\n            joblib.dump({"0": dict(frame_ids=ids, verts=joints)}, root / "cam.pkl")\n            rows = [dict(action=s, split=s, cam=c, GT=f"ground_truth_{s}", GT_path=str(root / "gt.npz"),\n                         PKL_path=str(root / "cam.pkl"), **{"file PKL": f"camera_{s}_{c}_012345"})\n                    for s in ("train", "val") for c in ("cam0", "cam1")]\n            with patch("train_v4.joints_from_vertices", side_effect=lambda v, r: v), \\\n                 patch("train_v4.associate", return_value=(ids, joints, {})):\n                pairs, _ = prepare(rows, None, root, CFG)\n            self.assertEqual(len(pairs), 8)\n            np.testing.assert_allclose(pairs[0]["target"], expected[:12], atol=1e-6)\n            np.testing.assert_allclose(pairs[2]["target"], expected[12:], atol=1e-6)\n\n    def test_prepare_reports_rejected_camera_and_keeps_valid_pairs(self):\n        mapping = load_keypoints3d_map("configs/keypoints3D_map.yml")\n        names = [k["name"] for k in mapping["keypoints"]]\n        joints = np.array([[skeleton(elbow_lift=0.01*t)[n] for n in names] for t in range(24)])\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            rows = []\n            for split, count in (("train", 3), ("val", 2)):\n                gt_id = f"{split}_ground_truth"\n                with (root / gt_id).open("wb") as stream:\n                    np.savez(stream, verts=joints)\n                for i in range(count):\n                    pkl_id = f"{split}_camera_{i}"\n                    length = 8 if split == "train" and i == 0 else len(joints)\n                    joblib.dump({"0": dict(frame_ids=np.arange(length), verts=joints[:length])}, root / pkl_id)\n                    rows.append(dict(action=split, split=split, cam=f"cam{i}",\n                                     GT=gt_id, **{"file PKL": pkl_id}))\n            report_path = root / "alignment.json"\n            with patch("train_v4.resolve_mounted", side_effect=lambda drive, value, mount, state=None: root / value), \\\n                 patch("train_v4.joints_from_vertices", side_effect=lambda vertices, reg: vertices):\n                pairs, reports = prepare(rows, None, root, CFG, report_path)\n                rejected = [r for r in reports if r["status"] == "rejected"]\n                self.assertEqual([(r["action"], r["cam"]) for r in rejected], [("train", "cam0")])\n                self.assertEqual(json.loads(report_path.read_text()), reports)\n                self.assertEqual({p["split"] for p in pairs}, {"train", "val"})\n                self.assertTrue(all("cam0" not in p["cameras"] for p in pairs if p["split"] == "train"))\n                with self.assertRaisesRegex(ValueError, "train: need at least two aligned cameras"):\n                    prepare([r for r in rows if not (r["split"] == "train" and r["cam"] == "cam2")],\n                            None, root, CFG, report_path)\n                self.assertEqual(json.loads(report_path.read_text())[0]["status"], "rejected")\n                bad_rows = [dict(action="bad", split="train", cam=f"cam{i}",\n                                 GT="train_ground_truth", **{"file PKL": "train_camera_0"})\n                            for i in range(2)]\n                pairs, reports = prepare(rows + bad_rows, None, root, CFG, report_path,\n                                         skip_unaligned_actions=True)\n                self.assertEqual({p["action"] for p in pairs}, {"train", "val"})\n                skipped = [r for r in reports if r["status"] == "skipped_action"]\n                self.assertEqual([r["action"] for r in skipped], ["bad"])\n                self.assertEqual(json.loads(report_path.read_text()), reports)\n                self.assertTrue(all(r["camera_frames"] == 8 and r["gt_frames"] == 24\n                                    for r in reports if r["status"] == "rejected"))\n                with self.assertRaisesRegex(ValueError, "No usable pairs for one of the splits"):\n                    prepare(bad_rows + [r for r in rows if r["split"] == "val"], None, root, CFG,\n                            report_path, skip_unaligned_actions=True)\n\n    def test_prepare_falls_back_to_download_and_cleans_temp_files(self):\n        mapping = load_keypoints3d_map("configs/keypoints3D_map.yml")\n        names = [k["name"] for k in mapping["keypoints"]]\n        joints = np.array([[skeleton(elbow_lift=0.01 * t)[n] for n in names] for t in range(24)])\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            gt_file = root / "raw_gt.npz"\n            with gt_file.open("wb") as stream:\n                np.savez(stream, verts=joints)\n            pkl_file = root / "raw_pkl.pkl"\n            joblib.dump({"0": dict(frame_ids=np.arange(len(joints)), verts=joints)}, pkl_file)\n            rows = [\n                dict(action=s, split=s, cam=c, GT=f"gt_{s}_0123456789", **{"file PKL": f"pkl_{s}_{c}_0123456"})\n                for s in ("train", "val") for c in ("cam0", "cam1")\n            ]\n            cache = root / "cache"\n            def fake_dl(drive, fid, dest):\n                dest.parent.mkdir(parents=True, exist_ok=True)\n                dest.write_bytes((gt_file if fid.startswith("gt_") else pkl_file).read_bytes())\n                return dest\n            with patch("train_v4.resolve_mounted", side_effect=FileNotFoundError("not mounted")), \\\n                 patch("train_v4._download_drive_file", side_effect=fake_dl) as dl_mock, \\\n                 patch("train_v4.joints_from_vertices", side_effect=lambda vertices, reg: vertices):\n                pairs, _ = prepare(rows, object(), cache, CFG, root / "alignment.json")\n            self.assertEqual(len(pairs), 4)\n            self.assertGreaterEqual(dl_mock.call_count, 6)\n            self.assertTrue((cache / "gt_train_0123456789.gt_joints.joblib").exists())\n            self.assertTrue((cache / "pkl_train_cam0_0123456.joints.joblib").exists())\n            self.assertFalse(list((cache / "_tmp").glob("*")))\n\n    def test_resolve_mounted_mydrive_shortcut_and_duplicates(self):\n        class FakeReq:\n            def __init__(self, payload):\n                self.payload = payload\n            def execute(self, num_retries=0):\n                if isinstance(self.payload, Exception):\n                    raise self.payload\n                return self.payload\n\n        class FakeFiles:\n            def __init__(self, db):\n                self.db = db\n            def get(self, fileId, fields="", supportsAllDrives=False):\n                return FakeReq(self.db.get(fileId, RuntimeError("404")))\n\n        class FakeDrive:\n            def __init__(self, db):\n                self._files = FakeFiles(db)\n            def files(self):\n                return self._files\n\n        with tempfile.TemporaryDirectory() as tmp:\n            mount = Path(tmp) / "drive"\n            (mount / "MyDrive/ds/seq").mkdir(parents=True)\n            f1 = mount / "MyDrive/ds/seq/wham.pkl"\n            f1_dup = mount / "MyDrive/ds/seq/wham (1).pkl"\n            f1.write_bytes(b"a")\n            f1_dup.write_bytes(b"b")\n            sc_dir = mount / ".shortcut-targets-by-id/shared_folder_12345"\n            sc_dir.mkdir(parents=True)\n            f2 = sc_dir / "sub/gt.npz"\n            f2.parent.mkdir(parents=True)\n            f2.write_bytes(b"gt")\n\n            db = {\n                "root": {"id": "root_0000000000"},\n                "ds_000000000000": {"id": "ds_000000000000", "name": "ds", "mimeType": "application/vnd.google-apps.folder", "parents": ["root_0000000000"]},\n                "seq_00000000000": {"id": "seq_00000000000", "name": "seq", "mimeType": "application/vnd.google-apps.folder", "parents": ["ds_000000000000"]},\n                "file_dup_000000": {"id": "file_dup_000000", "name": "wham.pkl", "parents": ["seq_00000000000"]},\n                "sub_00000000000": {"id": "sub_00000000000", "name": "sub", "mimeType": "application/vnd.google-apps.folder", "parents": ["shared_folder_12345"]},\n                "gt_shared_00000": {"id": "gt_shared_00000", "name": "gt.npz", "parents": ["sub_00000000000"]},\n                "missing_id_0000": {"id": "missing_id_0000", "name": "x.pkl", "parents": ["unshared_folder_99"]},\n                "orphan_id_00000": {"id": "orphan_id_00000", "name": "solo.npz", "owners": [{"emailAddress": "owner@lab.org"}]},\n            }\n            drive = FakeDrive(db)\n            (mount / "MyDrive/_v4_shortcuts").mkdir(parents=True)\n            (mount / "MyDrive/_v4_shortcuts/sc_file_000000").write_bytes(b"sc")\n            xattrs = {str(f1): "other_id_000000", str(f1_dup): "file_dup_000000"}\n            state = {"unmounted_targets": {}}\n            with patch("train_v4._xattr_drive_id", side_effect=lambda p: xattrs.get(str(p))):\n                # 0-API xattr scan works on an explicit dataset folder\n                scanned = index_mounted_files({"file_dup_000000"}, mount / "MyDrive/ds")\n                self.assertEqual(scanned["file_dup_000000"], f1_dup)\n                self.assertEqual(resolve_mounted(drive, "sc_file_000000", mount, state),\n                                 mount / "MyDrive/_v4_shortcuts/sc_file_000000")\n                self.assertEqual(resolve_mounted(drive, "file_dup_000000", mount, state), f1_dup)\n                self.assertEqual(resolve_mounted(drive, "gt_shared_00000", mount, state), f2)\n                with self.assertRaisesRegex(FileNotFoundError, "folders/unshared_folder_99"):\n                    resolve_mounted(drive, "missing_id_0000", mount, state)\n                with self.assertRaisesRegex(FileNotFoundError, "owner@lab.org"):\n                    resolve_mounted(drive, "orphan_id_00000", mount, state)\n                self.assertIn("orphan_id_00000", state["unmounted_targets"])\n                self.assertIn("unshared_folder_99", state["unmounted_targets"])\n\n    def test_untrimmed_camera_longer_than_gt_aligns_at_true_rate(self):\n        rng = np.random.default_rng(7)\n        names = pair()["names"]\n        gt = rng.normal(size=(40, len(names), 3))\n        extra = rng.normal(size=(60, len(names), 3))\n        cam = np.concatenate([gt @ np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]]) + 2, extra], axis=0)\n        ids, _, report = associate({"0": (np.arange(100), cam)}, gt, names)\n        self.assertEqual(report["ratio"], 1.0)\n        self.assertEqual(report["offset"], 0)\n        np.testing.assert_array_equal(ids, np.arange(40))\n\n    def test_alignment_rejects_insufficient_overlap(self):\n        names = pair()["names"]\n        gt = np.zeros((100, len(names), 3))\n        with self.assertRaisesRegex(AlignmentError, "No alignment covers"):\n            associate({"0": (np.arange(16), gt[:16])}, gt, names)\n\n    def test_leakage_and_duplicates_rejected(self):\n        rows = [dict(action=a, cam=c, split=s, GT=g, **{"file PKL": "some_id"})\n                for a,s,g in [("a","train","g1"),("b","val","g2")] for c in ["cam0","cam1"]]\n        validate_rows(rows)\n        bad = copy.deepcopy(rows)\n        bad[1]["split"] = "val"\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            validate_rows(bad)\n        with self.assertRaisesRegex(ValueError, "Duplicate"):\n            validate_rows(rows + rows[:1])\n        bad = copy.deepcopy(rows)\n        bad[2]["GT"] = "g1"\n        with self.assertRaisesRegex(ValueError, "leakage"):\n            validate_rows(bad)\n\n    def test_differentiable_fusion_matches_inference(self):\n        p = pair()\n        w = torch.nn.Parameter(torch.arange(8, dtype=torch.float32) / 10)\n        std = torch.ones(8)\n        actual = torch_fuse(p, w, std)\n        cfg = dict(CFG, weight_model=export_model(w.detach().numpy(), std.numpy()))\n        result, _ = fuse_partwise(p["M"], p["S"], cfg)\n        expected = centered(np.array([[f[n] for n in p["names"]] for f in result]), p["names"])\n        np.testing.assert_allclose(actual.detach().numpy(), expected, atol=2e-6)\n        actual.square().sum().backward()\n        self.assertTrue(torch.isfinite(w.grad).all())\n\n    def test_sequence_mpjpe_does_not_refit_fused_orientation(self):\n        p = pair("val")\n        yawed = [rotate(f, 25) for f in p["S"]]\n        with patch("train_v4.fuse_partwise", return_value=(yawed, {})):\n            metrics = validation([p], export_model(np.zeros(8), np.ones(8)), CFG)\n        self.assertLess(metrics["pa_mm"], 0.001)\n        self.assertGreater(metrics["mpjpe_mm"], 20)\n\n    def test_disabled_features_are_ignored_even_when_present_at_inference(self):\n        model = export_model(np.ones(8), np.ones(8), [True, False, False, False, True, True, True, True])\n        feats = np.zeros((2, 3, 8), dtype=np.float32)\n        expected = eval_weight_model(feats, model)\n        feats[..., 1:4] = 1000\n        np.testing.assert_array_equal(eval_weight_model(feats, model), expected)\n\n    def test_learned_weights_keep_visibility_and_depth_outlier_guards(self):\n        master = [skeleton(depth=4, elbow_lift=0.2) for _ in range(10)]\n        slave = [skeleton(depth=6) for _ in range(10)]\n        wm = dict(enabled=True, **export_model(np.zeros(8), np.ones(8)))\n        cfg = dict(CFG, weight_model=wm, multi_person={"enabled": True, "anti_swap_gate": False})\n        raw, _ = fuse_partwise(master, slave, dict(cfg, multi_person={"enabled": False}))\n        vis_m = [{"left_elbow": False, "left_wrist": False} for _ in master]\n        out, diag = fuse_partwise(master, slave, cfg, vis_master=vis_m)\n        self.assertGreater(diag["occluded_master_bones"], 0)\n        self.assertLess(np.linalg.norm(np.asarray(out[0]["left_wrist"]) - slave[0]["left_wrist"] + [0, 0, 2]),\n                        np.linalg.norm(np.asarray(raw[0]["left_wrist"]) - slave[0]["left_wrist"] + [0, 0, 2]))\n        _, diag = fuse_partwise(master, [skeleton(depth=18) for _ in master], cfg)\n        self.assertEqual(diag["depth_outlier_frames"], 10)\n        self.assertGreater(diag["master_weight_min"], 0.95)\n\n    def test_training_options_reject_invalid_values(self):\n        for options in ({"patience": 0}, {"patience": 1.5}, {"prior_l2": -1}, {"mpjpe_weight": float("nan")},\n                        {"naive_regret_weight": -1}, {"naive_margin_mm": float("inf")}):\n            with self.assertRaises(ValueError):\n                training_options(dict(CFG, training=options))\n\n    def test_naive_regret_changes_gradients_only_for_losing_frames(self):\n        pa = torch.tensor([0.02, 0.08], requires_grad=True)\n        mp = torch.tensor([0.07, 0.04], requires_grad=True)\n        loss, regret = naive_relative_loss(pa, mp, torch.tensor([0.05, 0.05]),\n                                           torch.tensor([0.06, 0.06]), training_options(CFG))\n        self.assertAlmostEqual(float(regret), 0.01625, places=7)\n        self.assertAlmostEqual(float(loss), 0.08, places=7)\n        loss.backward()\n        np.testing.assert_allclose(pa.grad.numpy(), [0.5, 1.0])\n        np.testing.assert_allclose(mp.grad.numpy(), [0.25, 0.125])\n\n    def test_naive_regret_margin_is_in_mm_and_can_be_disabled(self):\n        pa, mp = torch.tensor([0.049]), torch.tensor([0.04])\n        opts = training_options(dict(CFG, training={"naive_margin_mm": 2.0}))\n        loss, regret = naive_relative_loss(pa, mp, torch.tensor([0.05]), torch.tensor([0.06]), opts)\n        self.assertAlmostEqual(float(regret), 0.001, places=7)\n        opts["naive_regret_weight"] = 0.0\n        plain, _ = naive_relative_loss(pa, mp, torch.tensor([0.05]), torch.tensor([0.06]), opts)\n        self.assertAlmostEqual(float(loss - plain), 0.001, places=7)\n\n    def test_selection_requires_both_metrics_to_beat_naive(self):\n        naive = dict(pa_mm=50.0, mpjpe_mm=100.0)\n        opts = training_options(CFG)\n        compromise = selection_key(dict(pa_mm=40.0, mpjpe_mm=110.0), naive, opts)\n        both_better = selection_key(dict(pa_mm=49.0, mpjpe_mm=99.0), naive, opts)\n        self.assertLess(both_better, compromise)\n        self.assertGreater(both_better[1], compromise[1])\n        self.assertTrue(selection_key(naive, naive, opts)[0])\n        opts["naive_margin_mm"] = 1.0\n        self.assertTrue(selection_key(dict(pa_mm=49.0, mpjpe_mm=99.0), naive, opts)[0])\n\n    def test_naive_reference_copies_the_benchmark_smoothing_configuration(self):\n        pipeline = {"fusion": {"partwise": copy.deepcopy(CFG),\n                               "naive": {"master_weight": 0.5, "use_aotr_stability": True},\n                               "aotr": {"sequence_stability": {"enabled": True}}}}\n        before = copy.deepcopy(pipeline)\n        cfg = training_config(pipeline)\n        p = pair()\n        output = [{k: v.tolist() for k, v in f.items()} for f in p["M"]]\n        with patch("fusion_pipeline.aotr_stability.postprocess_aotr_3d", return_value=(output, {})) as smooth:\n            actual = naive_prediction(p, cfg)\n        self.assertEqual(smooth.call_args.kwargs["config"], {"enabled": True})\n        self.assertEqual(pipeline, before)\n        np.testing.assert_allclose(actual, np.array([[f[n] for n in p["names"]] for f in p["M"]]))\n\n    def test_frame_pa_errors_match_the_original_mean_loss(self):\n        p = pair()\n        pred = torch_fuse(p, torch.ones(8), torch.ones(8))\n        target = torch.as_tensor(p["target"])\n        errors = torch_pa_loss(pred, target, per_frame=True)\n        self.assertEqual(tuple(errors.shape), (len(p["M"]),))\n        self.assertAlmostEqual(float(errors.mean()), float(torch_pa_loss(pred, target)), places=7)\n        np.testing.assert_allclose(errors.detach().numpy(), pa_errors(pred.detach().numpy(), p["target"]), atol=1e-6)\n\n    def test_checkpoint_that_beats_both_metrics_is_kept_over_lower_objective(self):\n        def metrics(pa, mp):\n            return dict(pa_mm=pa, mpjpe_mm=mp, mono_pa_mm=60.0, mono_mpjpe_mm=120.0)\n\n        scores = [metrics(55, 110), metrics(55, 110), metrics(50, 100),\n                  metrics(40, 110), metrics(49, 99), metrics(35, 101)]\n        with tempfile.TemporaryDirectory() as tmp, patch("train_v4.validation", side_effect=scores):\n            history = fit([pair(), pair("val")], CFG, tmp, epochs=3)\n            best = json.loads((Path(tmp) / "best_weights.json").read_text())\n            self.assertEqual(best["epoch"], 2)\n            self.assertTrue(best["val_beats_naive"])\n            self.assertFalse(history[2]["val_beats_naive"])\n            self.assertTrue(history[2]["best_val_beats_naive"])\n\n    def test_unbeaten_naive_is_reported_and_training_still_stops_on_plateau(self):\n        naive = dict(pa_mm=0.0, mpjpe_mm=0.0, mono_pa_mm=20.0, mono_mpjpe_mm=30.0)\n        candidate = dict(pa_mm=10.0, mpjpe_mm=10.0, mono_pa_mm=20.0, mono_mpjpe_mm=30.0)\n        cfg = dict(CFG, training={"patience": 2})\n        with tempfile.TemporaryDirectory() as tmp, patch("train_v4.validation", side_effect=[candidate, candidate, naive] + [candidate] * 3):\n            history = fit([pair(), pair("val")], cfg, tmp, epochs=20)\n            self.assertEqual(len(history), 3)\n            best = json.loads((Path(tmp) / "best_weights.json").read_text())\n            self.assertFalse(best["val_beats_naive"])\n\n    def test_prior_inactive_features_and_early_stop_survive_resume(self):\n        cfg = dict(CFG, training={"patience": 2, "min_delta_mm": 1e6})\n        # A high learning rate exercises the non-negative depth projection.\n        with tempfile.TemporaryDirectory() as tmp:\n            history = fit([pair(), pair("val")], cfg, Path(tmp) / "first", epochs=10, lr=2)\n            self.assertEqual(len(history), 3)\n            saved = json.loads((Path(tmp) / "first/epoch_0003_weights.json").read_text())\n            self.assertGreaterEqual(saved["weights"][0], 0)\n            for i, active in enumerate(saved["active_features"]):\n                if not active:\n                    self.assertEqual(saved["std"][i], 1)\n                    self.assertEqual(saved["weights"][i], 0)\n            resumed = fit([pair(), pair("val")], cfg, Path(tmp) / "resume", epochs=10, lr=2,\n                          resume=Path(tmp) / "first/epoch_0003.pt")\n            self.assertEqual(history, resumed)\n            self.assertTrue((Path(tmp) / "resume/best_partwise.yml").exists())\n\n    def test_depth_outlier_guard_runs_in_features_and_without_multi_person(self):\n        master = [skeleton(depth=4) for _ in range(8)]\n        exploded = [skeleton(depth=18) for _ in range(8)]\n        f_ms, _ = extract_pair_features(master, exploded)\n        f_sm, _ = extract_pair_features(exploded, master)\n        self.assertTrue(np.all(f_ms[:, :, 0] == 3.0))\n        np.testing.assert_allclose(f_ms, -f_sm, atol=1e-6)\n        wm = dict(enabled=True, **export_model(np.zeros(8), np.ones(8)))\n        cfg = dict(CFG, weight_model=wm, multi_person={"enabled": False})\n        _, diag = fuse_partwise(master, exploded, cfg)\n        self.assertEqual(diag["depth_outlier_frames"], 8)\n        self.assertGreater(diag["master_weight_min"], 0.95)\n\n    def test_zero_prior_and_nonnegative_weights_constraint(self):\n        p = pair()\n        for t, frame in enumerate(p["M"]):\n            for value in frame.values():\n                value[2] += 0.1 * t\n        p["features"], _ = extract_pair_features(p["M"], p["S"])\n        p["directions"] = slave_directions(p["M"], p["S"], 4)\n        with tempfile.TemporaryDirectory() as tmp:\n            fit([p, pair("val")], CFG, Path(tmp) / "init", epochs=1, lr=1e-9)\n            init_model = json.loads((Path(tmp) / "init/epoch_0001_weights.json").read_text())\n            fit([p, pair("val")], CFG, Path(tmp) / "high_lr", epochs=3, lr=2.0)\n            high_model = json.loads((Path(tmp) / "high_lr/epoch_0003_weights.json").read_text())\n        self.assertTrue(init_model["active_features"][0])\n        np.testing.assert_allclose(init_model["weights"], np.zeros(8), atol=1e-6)\n        self.assertTrue(all(w >= 0.0 for w in high_model["weights"]))\n\n    def test_association_recovers_track_and_offset(self):\n        rng = np.random.default_rng(9)\n        p = pair()\n        names = p["names"]\n        gt = rng.normal(size=(45, len(names), 3))\n        correct = gt[7:37] @ np.array([[0,-1,0],[1,0,0],[0,0,1]]) + 4\n        wrong = rng.normal(size=correct.shape)\n        ids, joints, report = associate({"wrong": (np.arange(30), wrong),\n                                        "right": (np.arange(30), correct)}, gt, names)\n        self.assertEqual(report["track"], "right")\n        self.assertEqual(report["ratio"], 1)\n        self.assertEqual(report["offset"], 7)\n        np.testing.assert_array_equal(ids, np.arange(7,37))\n\n    def test_fragmented_tracks_and_double_rate_gt(self):\n        rng = np.random.default_rng(2)\n        names = pair()["names"]\n        gt = rng.normal(size=(60, len(names), 3))\n        joints = gt[::2] + 4\n        ids, actual, report = associate({"first": (np.arange(15), joints[:15]),\n                                        "second": (np.arange(15,30), joints[15:])}, gt, names)\n        self.assertEqual(report["ratio"], 2)\n        self.assertEqual(report["offset"], 0)\n        self.assertEqual(report["switches"], 1)\n        np.testing.assert_array_equal(ids, np.arange(0,60,2))\n        np.testing.assert_allclose(actual, joints)\n\n    def test_camera_phase_interpolation(self):\n        im, is_ = np.arange(0,20,2), np.arange(1,20,2)\n        jm, js = im[:,None,None] * np.ones((1,3,3)), is_[:,None,None] * np.ones((1,3,3))\n        ticks, master, slave = synchronize(im,jm,is_,js)\n        np.testing.assert_array_equal(ticks, np.arange(1,19))\n        np.testing.assert_allclose(master, slave)\n        np.testing.assert_allclose(master[:,0,0], ticks)\n\n    def test_checkpoint_best_and_resume_match_uninterrupted(self):\n        pairs = [pair(), pair("val")]\n        with tempfile.TemporaryDirectory() as tmp:\n            root = Path(tmp)\n            full = fit(pairs, CFG, root/"full", epochs=3)\n            fit(pairs, CFG, root/"part", epochs=1)\n            resumed = fit(pairs, CFG, root/"resume", epochs=3, resume=root/"part/epoch_0001.pt")\n            self.assertEqual(full, resumed)\n            best = json.loads((root/"resume/best_weights.json").read_text())\n            selected = min(resumed, key=lambda h: (not h["val_beats_naive"], h["val_objective_mm"]))\n            self.assertEqual(best["epoch"], selected["epoch"])\n            self.assertTrue((root/"resume/epoch_0003.pt").exists())\n            for key in ["weights","std"]:\n                a = json.loads((root/"full/epoch_0003_weights.json").read_text())[key]\n                b = json.loads((root/"resume/epoch_0003_weights.json").read_text())[key]\n                np.testing.assert_allclose(a,b)\n            with self.assertRaisesRegex(ValueError, "differs"):\n                fit(pairs, CFG, root/"bad", epochs=3, lr=0.2, resume=root/"part/epoch_0001.pt")\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'configs/pipeline.yml': 'runtime:\n  stage: visualization\n  clean_output: true\n  skip_unsynchronized_cameras: true\npreprocess:\n  offset_estimation:\n    max_frames: null\npose_export:\n  enabled: true\n  use_world_coords: false\ninputs:\n  cam1_pkl: /Users/thanhnt/Documents/research/optimization_model_monocular_3/inputs/drive/005_sword3/exo/cam03/output_wham/wham_output_cam03.pkl\n  cam2_pkl: /Users/thanhnt/Documents/research/optimization_model_monocular_3/inputs/drive/005_sword3/exo/cam04/output_wham/wham_output_cam04.pkl\n  camera1_video: /Users/thanhnt/Documents/research/optimization_model_monocular_3/inputs/drive/005_sword3/exo/cam03/images/harmony4D_005_sword3_cam03_ffmpeg.mp4\n  camera2_video: /Users/thanhnt/Documents/research/optimization_model_monocular_3/inputs/drive/005_sword3/exo/cam04/images/harmony4D_005_sword3_cam04_ffmpeg.mp4\n  ground_truth_dir: /Users/thanhnt/Documents/research/optimization_model_monocular_3/inputs/drive/005_sword3/processed_data/poses3d\n  target_person: aria01\npaths:\n  smpl_model: models/SMPL_NEUTRAL.pkl\n  segmentation: models/smpl_partSegmentation_mapping.pkl\n  keypoints3d_map: configs/keypoints3D_map.yml\n  keypoints2d_map: configs/keypoints2D_map.yml\n  j_regressor_3d: models/J_regressor_body25_extended.npy\n  preprocess_output_dir: output/preprocess_results\n  pose_output_dir: output/pose_results\n  fused_output_dir: output/fused_results\n  learnable_output_dir: output/learnable_results\n  learnable_extra_output_dir: output/learnable_extra_results\n  visualization_output_dir: output/visualize_results\n  evaluation_output_dir: output/evaluation_results\nfusion:\n  enabled: true\n  # Phương pháp dung hợp (tất cả đều cần aotr.enabled=true):\n  # - aotr    : trust-region lấy Master làm gốc (mặc định).\n  # - partwise: trộn đối xứng hướng xương, căn Slave theo từng phần thân (hông/vai),\n  #             trọng số theo độ sâu, giữ chiều dài xương Master. Xem MODEL_CHANGES.md.\n  # - naive   : BASELINE so sánh. Mỗi frame: căn Slave vào Master bằng một phép xoay Kabsch\n  #             (tâm hông), rồi lấy trung bình vị trí khớp. Không học.\n  # - adaptive_naive: ĐỐI CHỨNG cho AOTR. Cùng phép căn RANSAC thân, cùng ngưỡng và cùng bước\n  #             làm mượt với AOTR (lấy từ fusion.aotr), nhưng thay bộ tối ưu bằng trọng số đóng\n  #             cho từng xương: q = c^p * exp(-j/tau), w_Slave = q_S / (q_M + q_S),\n  #             u = normalize((1-w) u_M + w R21 u_S), dựng lại bằng FK với chiều dài xương Master.\n  #             c = confidence 2D của xương (min hai đầu mút có keypoint 2D trực tiếp);\n  #             j = độ rung hướng xương (sai phân bậc hai, trung bình ±h frame) / trung vị của xương đó.\n  #             Không dùng GT. Chỉ p và tau là tham số tự do: fit trên split train rồi đóng băng.\n  method: aotr\n  naive:\n    # Trọng số Master cố định; 0.5 = trung bình thuần. Không được tinh chỉnh (là baseline).\n    master_weight: 0.5\n    # true: qua cùng bước làm mượt của AOTR (fusion.aotr.sequence_stability: savgol + FK theo\n    # chiều dài xương Master), để so sánh không lẫn lợi ích làm mượt. false = baseline thô cũ.\n    use_aotr_stability: true\n  adaptive_naive:\n    confidence_power: 1.0     # p; 0 = bỏ confidence 2D\n    jitter_scale: 1.0         # tau, đơn vị trung vị độ rung; rất lớn = bỏ độ rung\n    jitter_half_window: 3     # h (frame)\n    missing_confidence: 0.5   # c khi xương/frame không có điểm 2D\n  partwise:\n    # Tham số logistic học MỘT LẦN trên split design (aria01, cam02/04/13/19) và đóng băng.\n    # w_Master = sigmoid(a + b * ((depth_M - depth_S) - mu) / sd), depth = z của tâm hông (m).\n    depth_logit:\n      a: -1.2388837544067565e-05\n      b: 1.6761106970863668\n      mu: 4.2041772210435745e-05\n      sd: 1.3685969107519123\n    # Trọng số v4 cũ, tạm tắt chờ benchmark lại (PA 12 khớp: learned 85.82, 50-50 86.49, mono 92.73 mm).\n    weight_model:\n      enabled: false\n      type: linear\n      feature_names:\n      - d_depth\n      - d_occ_joint\n      - d_occ_torso\n      - d_conf2d\n      - d_bone_jit\n      - d_cam_accel\n      - d_bone_distort\n      - d_facing\n      mean: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]\n      std:\n      - 2.0482571125030518\n      - 9.999999974752427e-07\n      - 9.999999974752427e-07\n      - 9.999999974752427e-07\n      - 1.4201602935791016\n      - 1.5395150184631348\n      - 0.005559795536100864\n      - 0.3992931544780731\n      weights:\n      - -0.4793398976325989\n      - 0.0\n      - 0.0\n      - 0.0\n      - 0.08805369585752487\n      - -0.4794311821460724\n      - -0.17094489932060242\n      - 0.36988016963005066\n    # Giá trị khởi đầu cho lần train mới; chọn trên validation, chưa xác nhận bằng benchmark.\n    training:\n      mpjpe_weight: 0.25\n      prior_l2: 0.001\n      naive_regret_weight: 1.0\n      naive_margin_mm: 0.0\n      patience: 10\n      min_delta_mm: 0.01\n    rotation_half_window: 4\n    multi_person:\n      enabled: true\n      visibility_weighted_rotation: true\n      master_occluded_logit_bias: 2.0\n      slave_occluded_logit_bias: 2.5\n      both_occluded_logit_bias: 0.5\n      anti_swap_gate: true\n      max_median_bone_angle_deg: 50.0\n      static_camera_orientation: false\n      orientation_smooth_half_window: 2\n    sequence_stability:\n      enabled: true\n      temporal_stability:\n        enabled: true\n        shape_weight: 2.5\n        residual_accel_weight: 1.0\n        residual_velocity_weight: 0.5\n        minimum_retention: 0.8\n        smooth_trajectories: true\n        smooth_window: 7\n        max_iter: 160\n      anatomical_calibration:\n        enabled: false\n  aotr:\n    enabled: true\n    missing_2d_mode: 3d_surrogate\n    three_d_consensus_weight: 0.12\n    three_d_slave_advantage_gain: 3.0\n    three_d_master_prior_weight: 0.5\n    three_d_temporal_weight: 0.15\n    three_d_jitter_epsilon: 0.06\n    three_d_occluded_master_reliability: 0.15\n    sequence_stability:\n      enabled: true\n      temporal_stability:\n        enabled: true\n        shape_weight: 2.5\n        residual_accel_weight: 1.0\n        residual_velocity_weight: 0.5\n        minimum_retention: 0.8\n        smooth_trajectories: true\n        smooth_window: 7\n        max_iter: 160\n      # Khóa chiều dài xương = Master (baseline) từng frame. Bật lại sẽ scale chi theo\n      # chiều cao ước lượng (sai ~12%) -> MBLE Hugging 22 -> 41 mm.\n      anatomical_calibration:\n        enabled: false\n    min_2d_confidence: 0.35\n    min_correspondences: 4\n    alignment_median_max_m: 0.12\n    alignment_q75_max_m: 0.18\n    ransac_threshold_m: 0.06\n    ransac_max_combos: 100\n    min_bone_direction_cos: -0.8\n    observability_ratio: 0.1\n    observed_prior_weight: 0.0001\n    unobserved_prior_weight: 0.0003 # cũ 0.003 - tinh chỉnh trên 003 aria01, kiểm tra trên 003 aria02\n    huber_delta_normalized: 0.016 #cũ 0.008\n    minimum_cost_reduction: 1.0e-07\n    trust_max_iter: 12\n    trust_initial_radius: 0.25\n    temporal_alignment_fallback: true\n    max_cross_view_bone_angle_deg: 50.0\n  selective_joint_fusion:\n    enabled: false\n    quality_margin_visible: 0.1\n    quality_margin_occluded: 0.06\n    max_displacement_visible: 0.12\n    max_displacement_occluded: 0.35\n    max_bone_relative_error: 0.15\n    minimum_slave_2d_support: 0.15\n    minimum_slave_belief: 0.1\n    alignment_median_max_m: 0.08\n    alignment_q75_max_m: 0.12\n    blend_strength: 0.35\n    optimize_selected: false\n  belief:\n    alpha: 0.001\n    beta: 0.8\n  occlusion:\n    enabled: true\n    tau: 0.05\n    multi_person: true\n    inter_person_tau: 0.15\n    inter_person_radius_m: 0.045\n    inter_person_min_hits: 3\n  uncertainty_gating:\n    consensus_blend_strength: 0.5\n    enabled: true\n    max_displacement: 0.35\n    min_confidence: 0.1\n    consensus_threshold: 0.08\n  master_protection:\n    enabled: true\n    confidence_margin: 0.15\n    default_mode: aggressive\n    pair_modes:\n      cam13->cam20: posed_fallback\n      cam17->cam20: posed_fallback\n      cam19->cam20: posed_fallback\n  angle_limits:\n    enabled: true\n    min_angle_deg: 30.0\n    max_angle_deg: 175.0\n  smoothing:\n    enabled: true\n    window_length: 5\n    polyorder: 2\n  ransac:\n    threshold: 0.05\n    max_combos: 500\n  optimization:\n    enabled: true\n    regularization: true\n    regularization_lambda: 1.0\n    temporal_lambda: 2.0\n    max_iter: 35\nlearnable:\n  enabled: false\n  checkpoint: models/best_ckpt.pth.tar\nlearnable_extra:\n  enabled: false\nvisualization:\n  enabled: false\n  target_fps: 20\n  dpi: 100\n  max_frames: null\n  cameras:\n  - camera1\n  - camera2\nevaluation:\n  enabled: true\n  testcase_name: null\n  # Loại các frame mà track WHAM bám nhầm người (tráo ID) khỏi phần đánh giá.\n  # Tạo bằng: python build_identity_mask.py configs/brute_force.yml configs/identity_mask_hugging_002.json\n  # Bộ khớp chấm điểm: coco17 (Harmony4D 3D keypoints: 12 khớp thân + mũi/mắt/tai) | priority1 (cũ, 12 khớp)\n  joint_set: coco17\n  # Căn chỉnh MPJPE: sequence = gốc tâm hông + MỘT phép xoay cố định mỗi camera cho cả chuỗi\n  # (thay cho extrinsics khi không có); frame = xoay lại từng frame (cũ, dễ dãi hơn).\n  mpjpe_alignment: sequence\n  # 043_Grappling2: nhãn người tạo bằng python -m preprocessing_data.label_persons (xem README).\n  identity_mask:\n    path:\n      - configs/identity_mask_hugging_002.json\n      - configs/identity_mask_grappling2_043.json\n      - configs/identity_mask_005_sword3.json\n    margin_mm: 10.0\n  metrics:\n    pa_mpjpe: true\n    mpjpe: true\n    pck: false\n    pck_threshold_mm: 150.0\n    mble: true\n    accel: true\n'}
for name, text in SNAPSHOTS.items():
    path = REPO_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
import importlib
if "train_v4" in sys.modules:
    importlib.reload(sys.modules["train_v4"])
run_live([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_train_v4.py", "-v"])


In [ ]:
#@title 3. Mount Google Drive, đọc Google Sheet & cấu hình lưu checkpoint
import logging, re
from google.colab import auth, drive
import google.auth, gspread
from train_v4 import validate_rows

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

DRIVE_MOUNT = "/content/drive"  #@param {type:"string"}
DRIVE_DATA_DIR = ""  #@param {type:"string"}
SPREADSHEET_ID = "11OipPLik-aXw0GmrTfEwg6wsAQYEmvGEV8VRlyry4Rg"  #@param {type:"string"}
TAB = "v4 22/11/26"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/partwise_v4/run_004"  #@param {type:"string"}
EPOCHS = 50  #@param {type:"integer"}
LEARNING_RATE = 0.01  #@param {type:"number"}
RESUME = True  #@param {type:"boolean"}
SKIP_UNALIGNED_ACTIONS = True  #@param {type:"boolean"}

SCOPES = [
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
]
auth.authenticate_user()
drive.mount(DRIVE_MOUNT, force_remount=False)
credentials, _ = google.auth.default(scopes=SCOPES)
client = gspread.authorize(credentials)

OUTPUT = Path(OUTPUT_DIR.strip())
CACHE = OUTPUT / "joints_cache"
OUTPUT.mkdir(parents=True, exist_ok=True)

sh = client.open_by_key(SPREADSHEET_ID.strip())
try:
    sheet = sh.worksheet(TAB.strip())
except gspread.WorksheetNotFound:
    sheet = sh.worksheet(TAB.strip().replace("-", " "))
values = sheet.get(f"A1:O{sheet.row_count}")
headers = [h.strip() for h in values[0]]
rows = []
for raw_row in values[1:]:
    if not any(str(c).strip() for c in raw_row):
        continue
    d = dict(zip(headers, [str(c).strip() for c in raw_row] + [""] * (len(headers) - len(raw_row))))
    d["action"] = re.sub(r"_cam\d+$", "", d.get("action", ""))
    rows.append(d)

groups = validate_rows(rows)
print(f"Loaded tab '{sheet.title}' | Rows: {len(rows)} | Train sequences: {list(groups.values()).count('train')} | Val sequences: {list(groups.values()).count('val')}", flush=True)

MANIFEST = OUTPUT / "manifest_input.json"
if (OUTPUT / "last.json").exists() and MANIFEST.exists():
    old_clean = [{k: v for k, v in r.items() if k not in ("GT_path", "PKL_path")}
                 for r in json.loads(MANIFEST.read_text(encoding="utf-8"))]
    new_clean = [{k: v for k, v in r.items() if k not in ("GT_path", "PKL_path")} for r in rows]
    if old_clean != new_clean:
        raise ValueError("Dữ liệu Sheet đã thay đổi so với checkpoint cũ; hãy đổi OUTPUT_DIR sang thư mục mới.")
MANIFEST.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
print("Output Drive folder:", OUTPUT)


In [ ]:
#@title 4a. Chuẩn bị dữ liệu & kiểm tra căn chỉnh (Alignment)
import pandas as pd

prepare_cmd = [sys.executable, "-u", "train_v4.py", "--manifest", str(MANIFEST),
               "--drive-mount", DRIVE_MOUNT, "--data-dir", DRIVE_DATA_DIR.strip(),
               "--cache", str(CACHE), "--output", str(OUTPUT), "--prepare-only"]
if SKIP_UNALIGNED_ACTIONS:
    prepare_cmd.append("--skip-unaligned-actions")
run_live(prepare_cmd)

alignment = pd.DataFrame(json.loads((OUTPUT / "alignment.json").read_text(encoding="utf-8")))
print("Tổng hợp trạng thái camera/sequence:", alignment["status"].value_counts().to_dict())
flagged = alignment[
    alignment["status"].isin(["rejected", "skipped_action"])
    | (alignment["coverage"].fillna(1.0) < 0.8)
    | (alignment["pa_mm"].fillna(0.0) > 80.0)
    | (alignment.get("switches", pd.Series(0, index=alignment.index)).fillna(0) > 0)
]
if not flagged.empty:
    print("Các camera/sequence cần xem lại (bị loại, phủ <80% GT, PA >80mm, hoặc chuyển track):")
    display(flagged)
else:
    print("Không có cờ cảnh báo tự động; vẫn cần kiểm tra alignment và danh tính track.")


In [ ]:
#@title 4b. Chạy Train & lưu checkpoint từng epoch trên Google Drive
import hashlib, shutil, torch, yaml
from train_v4 import fit_signature, training_config

# Tính đúng chữ ký cấu hình của train_v4.py để kiểm tra tương thích trước khi resume
cfg_pw = training_config(yaml.safe_load((REPO_DIR / "configs/pipeline.yml").read_text(encoding="utf-8")))
stable_rows = [{k: v for k, v in r.items() if k not in ("GT_path", "PKL_path")}
               for r in json.loads(MANIFEST.read_text(encoding="utf-8"))]
cur_identity = hashlib.sha256(
    json.dumps(stable_rows, sort_keys=True, ensure_ascii=False).encode()
    + (REPO_DIR / "models/J_regressor_body25_extended.npy").read_bytes()
    + (REPO_DIR / "train_v4.py").read_bytes()
    + (REPO_DIR / "fusion_pipeline/partwise.py").read_bytes()
    + (REPO_DIR / "fusion_pipeline/naive.py").read_bytes()
    + (REPO_DIR / "fusion_pipeline/aotr_stability.py").read_bytes()
    + (REPO_DIR / "configs/keypoints3D_map.yml").read_bytes()
    + json.dumps(cfg_pw, sort_keys=True).encode()
    + str(bool(SKIP_UNALIGNED_ACTIONS)).encode()
).hexdigest()
cur_sig = fit_signature(cur_identity, cfg_pw, float(LEARNING_RATE))


def _probe_ckpt(folder):
    lp = folder / "last.json"
    if not lp.exists():
        return None, None, None
    info = json.loads(lp.read_text(encoding="utf-8"))
    pt = folder / info.get("checkpoint", "")
    if not pt.exists():
        return None, None, None
    sig = info.get("signature") or torch.load(pt, map_location="cpu", weights_only=False).get("signature")
    return pt, sig, info.get("epoch")


pt, sig, ep = _probe_ckpt(OUTPUT)
has_pts = any(OUTPUT.glob("epoch_*.pt"))
if not has_pts:
    ACTIVE_OUT, resume_pt = OUTPUT, None
elif RESUME and pt is not None and sig == cur_sig:
    ACTIVE_OUT, resume_pt = OUTPUT, pt
else:
    # Khác lr/config hoặc tắt RESUME: tự tách sang thư mục con và tái sử dụng dữ liệu đã căn chỉnh
    sub = OUTPUT / f"lr_{float(LEARNING_RATE):g}"
    pt2, sig2, ep2 = _probe_ckpt(sub)
    if RESUME and pt2 is not None and sig2 == cur_sig:
        ACTIVE_OUT, resume_pt, ep = sub, pt2, ep2
    elif not any(sub.glob("epoch_*.pt")):
        ACTIVE_OUT, resume_pt = sub, None
    else:
        k = 2
        while any((OUTPUT / f"lr_{float(LEARNING_RATE):g}_run{k}").glob("epoch_*.pt")):
            k += 1
        ACTIVE_OUT, resume_pt = OUTPUT / f"lr_{float(LEARNING_RATE):g}_run{k}", None
    ACTIVE_OUT.mkdir(parents=True, exist_ok=True)
    prep_src = OUTPUT / f"prepared_{cur_identity}.joblib"
    prep_dst = ACTIVE_OUT / prep_src.name
    if prep_src.exists() and not prep_dst.exists():
        shutil.copy2(prep_src, prep_dst)
    for aux in ("alignment.json", "manifest.json", "protocol.json"):
        if (OUTPUT / aux).exists() and not (ACTIVE_OUT / aux).exists():
            shutil.copy2(OUTPUT / aux, ACTIVE_OUT / aux)
    print(f"Khác lr/cấu hình so với checkpoint gốc -> lưu tại: {ACTIVE_OUT} (tái sử dụng dữ liệu đã căn chỉnh)")

train_cmd = [sys.executable, "-u", "train_v4.py", "--manifest", str(MANIFEST),
             "--drive-mount", DRIVE_MOUNT, "--cache", str(CACHE),
             "--output", str(ACTIVE_OUT), "--epochs", str(EPOCHS), "--lr", str(LEARNING_RATE)]
if SKIP_UNALIGNED_ACTIONS:
    train_cmd.append("--skip-unaligned-actions")
if resume_pt is not None:
    print(f"Resume từ checkpoint: {resume_pt} (epoch {ep})")
    train_cmd += ["--resume", str(resume_pt)]

run_live(train_cmd)


In [ ]:
#@title 5. Chọn Best Epoch, xem bộ tham số tốt nhất & vẽ đường học
import matplotlib.pyplot as plt
import pandas as pd

RUN_OUT = globals().get("ACTIVE_OUT", OUTPUT)
history = json.loads((RUN_OUT / "history.json").read_text(encoding="utf-8"))
best = json.loads((RUN_OUT / "best_weights.json").read_text(encoding="utf-8"))
saved_pts = sorted(RUN_OUT.glob("epoch_*.pt"))

print(f"Đã lưu {len(saved_pts)} epoch checkpoints tại: {RUN_OUT}")
print(f"★ Best Epoch          : {best['epoch']} / {len(history)}")
print(f"  Val MPJPE sequence  : {best['val_sequence_mpjpe_mm']:.2f} mm")
print(f"  Val objective       : {best['val_objective_mm']:.2f} mm")
print(f"  Val PA-MPJPE (Best) : {best['val_mpjpe_mm']:.2f} mm")
if "equal_val_mpjpe_mm" in best:
    print(f"  Val PA-MPJPE (50-50): {best['equal_val_mpjpe_mm']:.2f} mm")
print(f"  Val PA-MPJPE (Mono) : {best['baseline_val_mpjpe_mm']:.2f} mm")
print(f"  Naive PA / MPJPE    : {best['naive_val_mpjpe_mm']:.2f} / {best['naive_val_sequence_mpjpe_mm']:.2f} mm")
print(f"  Gain vs naive PA/MP : {best['val_pa_gain_vs_naive_mm']:.2f} / {best['val_mpjpe_gain_vs_naive_mm']:.2f} mm (dương = tốt hơn)")
print(f"  Thắng cả PA và MP   : {best['val_beats_naive']}")
if not best["val_beats_naive"]:
    print("Chưa có checkpoint thắng naive ở cả hai chỉ số validation; file best là ứng viên để kiểm tra.")
print(f"  File tham số tốt nhất: {RUN_OUT / 'best_weights.json'}")
print(f"  File patch YAML     : {RUN_OUT / 'best_partwise.yml'}")

param_df = pd.DataFrame({
    "feature": best["feature_names"],
    "weight": [round(w, 6) for w in best["weights"]],
    "train_std": [round(s, 6) for s in best["std"]],
})
display(param_df)

epochs_x = [h["epoch"] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs_x, [h["train_mpjpe_mm"] for h in history], label="Train PA-MPJPE")
plt.plot(epochs_x, [h["val_mpjpe_mm"] for h in history], label="Val PA-MPJPE (partwise learned)")
if "equal_val_mpjpe_mm" in history[0]:
    plt.axhline(history[0]["equal_val_mpjpe_mm"], color="gray", linestyle="--", label="Val 50-50 (w=0.5)")
plt.axhline(history[0]["depth_val_mpjpe_mm"], color="green", linestyle=":", label="Val depth_logit")
plt.axhline(history[0]["naive_val_mpjpe_mm"], color="orange", linestyle="--", label="Val naive")
plt.scatter([best["epoch"]], [best["val_mpjpe_mm"]], color="red", zorder=5, label=f"Selected checkpoint (epoch {best['epoch']})")
plt.xlabel("Epoch")
plt.ylabel("PA-MPJPE (mm)")
plt.title("Quá trình huấn luyện bộ trọng số partwise")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
plt.figure(figsize=(7, 4))
plt.plot(epochs_x, [h["val_sequence_mpjpe_mm"] for h in history], label="Val MPJPE learned")
plt.axhline(history[0]["depth_val_sequence_mpjpe_mm"], label="Val MPJPE depth_logit", linestyle="--")
plt.axhline(history[0]["naive_val_sequence_mpjpe_mm"], label="Val MPJPE naive", linestyle="-.")
plt.axhline(history[0]["baseline_val_sequence_mpjpe_mm"], label="Val MPJPE mono", linestyle=":")
plt.xlabel("Epoch")
plt.ylabel("Sequence MPJPE (mm), 12 body joints")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
#@title 6. Ghép bộ tham số tốt nhất vào pipeline_best.yml để suy luận / benchmark
import yaml

RUN_OUT = globals().get("ACTIVE_OUT", OUTPUT)
base = yaml.safe_load((REPO_DIR / "configs/pipeline.yml").read_text(encoding="utf-8"))
patch = yaml.safe_load((RUN_OUT / "best_partwise.yml").read_text(encoding="utf-8"))
base["fusion"]["method"] = "partwise"
base["fusion"]["partwise"] = patch["fusion"]["partwise"]
out_cfg = RUN_OUT / "pipeline_best.yml"
out_cfg.write_text(yaml.safe_dump(base, sort_keys=False, allow_unicode=True), encoding="utf-8")
print("Cấu hình thử nghiệm: cần benchmark coco17 trước khi dùng chính thức.")
print("Đã xuất cấu hình đầy đủ (nhúng sẵn trọng số, dùng được ngay cả trên Colab lẫn máy local):")
print(" ->", out_cfg)
print("\nĐoạn cấu hình weight_model tốt nhất:")
print(yaml.safe_dump({"weight_model": patch["fusion"]["partwise"]["weight_model"]}, sort_keys=False))
